# Analyse des Interactions entre Analogues de Chaînes Latérales dans une Membrane POPC

## Objectif
Ce notebook implémente une analyse quantitative des interactions entre analogues de chaînes latérales d'acides aminés dans une membrane lipidique de POPC. L'objectif est de caractériser la distribution des analogues au sein de la membrane et de contrôler l'effet potentiel des interactions inter-analogues sur les calculs d'énergie libre d'insertion.

## Méthodologie
Deux définitions du contact sont implémentées et comparées :
1. **Définition par distance atomique minimale** : Contact défini par $d_{\min} = \min_{i,j} \|\mathbf{r}_i^A - \mathbf{r}_j^B\|$ entre atomes lourds
2. **Définition par centre de masse** : Contact défini par $d_{\text{COM}} = \|\mathbf{R}_{\text{COM}}^A - \mathbf{R}_{\text{COM}}^B\|$ avec cutoffs dépendant de l'analogue

## Types d'interactions analysées
- Interactions de van der Waals (cutoff typique : 4.0 Å)
- Liaisons hydrogène (cutoff typique : 3.5 Å)
- Interactions hydrophobes (cutoff typique : 4.5 Å)
- Interactions π–π (cutoff typique : 5.0 Å)

## Données d'entrée
Fichiers PDB situés dans `supp_files/output_systems/sc*/*.pdb` correspondant à des frames indépendantes de trajectoires MD.

## 1. Import des bibliothèques requises

Importation des bibliothèques standards pour l'analyse moléculaire :
- **MDAnalysis** : Lecture et manipulation de structures moléculaires
- **Biopython** : Analyse structurale et détection de contacts
- **NumPy/SciPy** : Calculs numériques et analyse statistique
- **Matplotlib** : Visualisation des résultats

### ⚠️ Résolution des problèmes de compatibilité

Si vous rencontrez l'erreur `ValueError: numpy.dtype size changed`, cela indique une incompatibilité binaire entre numpy et h5py. 

**Solution recommandée** : Exécutez ces commandes dans un terminal :

```bash
# Option 1 : Réinstaller h5py avec pip
pip uninstall h5py -y
pip install --no-cache-dir h5py

# Option 2 : Si vous utilisez conda
conda install -c conda-forge h5py --force-reinstall

# Option 3 : Mettre à jour numpy et h5py ensemble
pip install --upgrade numpy h5py
```

Après avoir exécuté l'une de ces commandes, redémarrez le kernel du notebook (Kernel → Restart Kernel).

In [3]:
# =============================================================================
# Import des bibliothèques
# =============================================================================
import os
import glob
import json
import warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.spatial import distance
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns

# MDAnalysis pour l'analyse de trajectoires
import MDAnalysis as mda
from MDAnalysis.analysis import distances as mda_distances

# Biopython pour l'analyse structurale
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import is_aa

# Configuration des warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Configuration de matplotlib pour des figures publication-ready
plt.rcParams.update({
    'figure.figsize': (10, 8),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 11,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'axes.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Palette de couleurs cohérente pour les analyses
COLORS = {
    'vdw': '#1f77b4',      # Bleu
    'hbond': '#2ca02c',     # Vert
    'hydrophobic': '#ff7f0e',  # Orange
    'pipi': '#9467bd',      # Violet
    'any': '#d62728',       # Rouge
}

print("✓ Bibliothèques importées avec succès")
print(f"  - MDAnalysis version: {mda.__version__}")
print(f"  - NumPy version: {np.__version__}")
print(f"  - Pandas version: {pd.__version__}")

✓ Bibliothèques importées avec succès
  - MDAnalysis version: 2.10.0
  - NumPy version: 2.3.5
  - Pandas version: 2.3.3


## 2. Configuration et découverte des données

### 2.1 Chemins et paramètres globaux
Configuration des chemins vers les données et des paramètres d'analyse.

In [4]:
# =============================================================================
# Configuration des chemins et paramètres globaux
# =============================================================================

# Chemin racine du projet (relatif au notebook)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]  # Remonte à la racine du projet

# Chemins vers les données
DATA_DIR = PROJECT_ROOT / "supp_files" / "output_systems"
OUTPUT_DIR = NOTEBOOK_DIR / "results"
FIGURES_DIR = NOTEBOOK_DIR / "figures"

# Créer les répertoires de sortie s'ils n'existent pas
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# =============================================================================
# Paramètres d'analyse
# =============================================================================

# Cutoffs par défaut pour les différents types d'interactions (en Ångströms)
DEFAULT_CUTOFFS = {
    'vdw': 4.0,           # van der Waals: somme des rayons VdW + tolérance
    'hbond': 3.5,         # Liaison H: distance donneur-accepteur typique
    'hydrophobic': 4.5,   # Hydrophobe: distance C-C typique
    'pipi': 5.0,          # π-π: distance entre centroïdes aromatiques
}

# Plages de cutoff à explorer pour l'analyse paramétrique
CUTOFF_RANGES = {
    'min_heavy': np.arange(2.5, 8.1, 0.25),   # Distance min atomes lourds
    'com': np.arange(4.0, 15.1, 0.5),          # Distance entre COM
}

# Définition des atomes pour chaque type d'interaction
ATOM_DEFINITIONS = {
    # Atomes donneurs/accepteurs de liaisons H (N, O avec H attachés ou électrons libres)
    'hbond_donors': {'N', 'O'},
    'hbond_acceptors': {'N', 'O', 'S'},
    
    # Atomes hydrophobes (carbones aliphatiques et certains soufres)
    'hydrophobic': {'C'},  # Seuls les C non polaires seront sélectionnés
    
    # Atomes aromatiques typiques des cycles
    'aromatic_atoms': {'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ', 'CH2', 'NE1'},
}

# Correspondance des codes de dossiers vers les noms de résidus
# Format: dossier -> (nom_résidu, nom_acide_aminé, type)
ANALOGUE_INFO = {
    'sca': ('SCA', 'Alanine', 'aliphatic'),
    'scc': ('SCC', 'Cysteine', 'polar'),
    'sccm': ('SCCM', 'Cysteine-methyl', 'polar'),
    'scd': ('SCD', 'Aspartate', 'charged'),
    'scdn': ('SCDN', 'Aspartate-neutral', 'polar'),
    'sce': ('SCE', 'Glutamate', 'charged'),
    'scen': ('SCEN', 'Glutamate-neutral', 'polar'),
    'scf': ('SCF', 'Phenylalanine', 'aromatic'),
    'schd': ('SCHD', 'Histidine-delta', 'aromatic'),
    'sche': ('SCHE', 'Histidine-epsilon', 'aromatic'),
    'schp': ('SCHP', 'Histidine-protonated', 'charged'),
    'sci': ('SCI', 'Isoleucine', 'aliphatic'),
    'sck': ('SCK', 'Lysine', 'charged'),
    'sckn': ('SCKN', 'Lysine-neutral', 'polar'),
    'scl': ('SCL', 'Leucine', 'aliphatic'),
    'scm': ('SCM', 'Methionine', 'aliphatic'),
    'scn': ('SCN', 'Asparagine', 'polar'),
    'scp': ('SCP', 'Proline', 'aliphatic'),
    'scq': ('SCQ', 'Glutamine', 'polar'),
    'scr': ('SCR', 'Arginine', 'charged'),
    'scrn': ('SCRN', 'Arginine-neutral', 'polar'),
    'scs': ('SCS', 'Serine', 'polar'),
    'sct': ('SCT', 'Threonine', 'polar'),
    'scv': ('SCV', 'Valine', 'aliphatic'),
    'scw': ('SCW', 'Tryptophan', 'aromatic'),
    'scy': ('SCY', 'Tyrosine', 'aromatic'),
    'scym': ('SCYM', 'Tyrosine-methyl', 'aromatic'),
}

# Rayon de giration approximatif pour les cutoffs COM dépendant de l'analogue (Å)
# Valeurs approximatives basées sur la taille des chaînes latérales
ANALOGUE_RADII = {
    'sca': 2.0,   # Alanine - petite
    'scc': 2.5,   # Cysteine
    'sccm': 3.0,  # Cysteine-methyl
    'scd': 3.5,   # Aspartate
    'scdn': 3.5,
    'sce': 4.0,   # Glutamate
    'scen': 4.0,
    'scf': 4.5,   # Phenylalanine - cycle aromatique
    'schd': 4.5,  # Histidine
    'sche': 4.5,
    'schp': 4.5,
    'sci': 3.5,   # Isoleucine
    'sck': 5.0,   # Lysine - longue chaîne
    'sckn': 5.0,
    'scl': 3.5,   # Leucine
    'scm': 4.0,   # Methionine
    'scn': 3.5,   # Asparagine
    'scp': 3.0,   # Proline
    'scq': 4.0,   # Glutamine
    'scr': 5.5,   # Arginine - très longue
    'scrn': 5.5,
    'scs': 2.5,   # Serine
    'sct': 3.0,   # Threonine
    'scv': 3.0,   # Valine
    'scw': 5.0,   # Tryptophan - grand cycle
    'scy': 4.5,   # Tyrosine
    'scym': 4.5,
}

print(f"✓ Configuration chargée")
print(f"  - Répertoire des données: {DATA_DIR}")
print(f"  - Répertoire de sortie: {OUTPUT_DIR}")
print(f"  - {len(ANALOGUE_INFO)} types d'analogues définis")

✓ Configuration chargée
  - Répertoire des données: /Users/valentin/Dropbox/SC_article/Bories_and_Lague_2025/supp_files/output_systems
  - Répertoire de sortie: /Users/valentin/Dropbox/SC_article/Bories_and_Lague_2025/figures/data/mono_data/cutoff_research/results
  - 27 types d'analogues définis


### 2.2 Découverte automatique des fichiers PDB

Exploration du répertoire `output_systems` pour identifier tous les systèmes disponibles et leurs fichiers PDB associés.

In [5]:
# =============================================================================
# Découverte des fichiers PDB disponibles
# =============================================================================

def discover_pdb_files(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Découvre tous les fichiers PDB dans les sous-répertoires sc*.
    
    Parameters
    ----------
    data_dir : Path
        Répertoire racine contenant les sous-dossiers sc*
        
    Returns
    -------
    Dict[str, List[Path]]
        Dictionnaire {nom_système: [liste_fichiers_pdb]}
    """
    systems = {}
    
    # Recherche des dossiers sc*
    sc_dirs = sorted(data_dir.glob("sc*"))
    
    for sc_dir in sc_dirs:
        if not sc_dir.is_dir():
            continue
            
        system_name = sc_dir.name
        pdb_files = sorted(sc_dir.glob("*.pdb"))
        
        # Filtrer les fichiers non vides
        valid_files = [f for f in pdb_files if f.stat().st_size > 1000]
        
        if valid_files:
            systems[system_name] = valid_files
            
    return systems


def get_system_info(systems: Dict[str, List[Path]]) -> pd.DataFrame:
    """
    Génère un tableau récapitulatif des systèmes découverts.
    
    Parameters
    ----------
    systems : Dict[str, List[Path]]
        Dictionnaire des systèmes et fichiers
        
    Returns
    -------
    pd.DataFrame
        Tableau avec informations sur chaque système
    """
    data = []
    for system_name, files in systems.items():
        info = ANALOGUE_INFO.get(system_name, (system_name.upper(), 'Unknown', 'unknown'))
        radius = ANALOGUE_RADII.get(system_name, 3.5)
        
        data.append({
            'Système': system_name,
            'Résidu': info[0],
            'Acide aminé': info[1],
            'Type': info[2],
            'Nb fichiers': len(files),
            'Rayon (Å)': radius,
        })
    
    return pd.DataFrame(data)


# Découverte des systèmes
print("Découverte des fichiers PDB...")
pdb_systems = discover_pdb_files(DATA_DIR)

# Affichage du résumé
systems_df = get_system_info(pdb_systems)
print(f"\n✓ {len(pdb_systems)} systèmes découverts avec {sum(len(f) for f in pdb_systems.values())} fichiers PDB au total\n")

# Affichage du tableau
display(systems_df.style.set_caption("Systèmes d'analogues disponibles"))

Découverte des fichiers PDB...

✓ 27 systèmes découverts avec 81 fichiers PDB au total



,Système,Résidu,Acide aminé,Type,Nb fichiers,Rayon (Å)
0,sca,SCA,Alanine,aliphatic,3,2.000000
1,scc,SCC,Cysteine,polar,3,2.500000
2,sccm,SCCM,Cysteine-methyl,polar,3,3.000000
3,scd,SCD,Aspartate,charged,3,3.500000
4,scdn,SCDN,Aspartate-neutral,polar,3,3.500000
5,sce,SCE,Glutamate,charged,3,4.000000
6,scen,SCEN,Glutamate-neutral,polar,3,4.000000
7,scf,SCF,Phenylalanine,aromatic,3,4.500000
8,schd,SCHD,Histidine-delta,aromatic,3,4.500000
9,sche,SCHE,Histidine-epsilon,aromatic,3,4.500000


## 3. Structures de données moléculaires

### 3.1 Classe Analogue
Définition d'une classe pour représenter un analogue de chaîne latérale avec ses propriétés atomiques et méthodes de calcul géométrique.

In [6]:
# =============================================================================
# Structures de données pour les analogues
# =============================================================================

@dataclass
class Atom:
    """Représentation d'un atome."""
    name: str
    element: str
    coords: np.ndarray
    index: int
    mass: float = 1.0
    
    def distance_to(self, other: 'Atom') -> float:
        """Calcule la distance euclidienne vers un autre atome."""
        return np.linalg.norm(self.coords - other.coords)


@dataclass 
class Analogue:
    """
    Représentation d'un analogue de chaîne latérale.
    
    Attributes
    ----------
    resid : int
        Numéro de résidu
    resname : str
        Nom du résidu (ex: SCY, SCF)
    atoms : List[Atom]
        Liste de tous les atomes
    """
    resid: int
    resname: str
    atoms: List[Atom] = field(default_factory=list)
    
    # Propriétés cachées calculées à la demande
    _heavy_atoms: Optional[List[Atom]] = field(default=None, repr=False)
    _aromatic_atoms: Optional[List[Atom]] = field(default=None, repr=False)
    _com: Optional[np.ndarray] = field(default=None, repr=False)
    _aromatic_com: Optional[np.ndarray] = field(default=None, repr=False)
    
    @property
    def heavy_atoms(self) -> List[Atom]:
        """Retourne uniquement les atomes lourds (non-H)."""
        if self._heavy_atoms is None:
            self._heavy_atoms = [a for a in self.atoms if a.element != 'H']
        return self._heavy_atoms
    
    @property
    def heavy_coords(self) -> np.ndarray:
        """Coordonnées des atomes lourds sous forme de tableau numpy."""
        return np.array([a.coords for a in self.heavy_atoms])
    
    @property
    def all_coords(self) -> np.ndarray:
        """Coordonnées de tous les atomes."""
        return np.array([a.coords for a in self.atoms])
    
    @property
    def center_of_mass(self) -> np.ndarray:
        """
        Calcule le centre de masse de l'analogue.
        Utilise les masses atomiques standards.
        """
        if self._com is None:
            masses = np.array([ATOMIC_MASSES.get(a.element, 12.0) for a in self.heavy_atoms])
            coords = self.heavy_coords
            self._com = np.average(coords, axis=0, weights=masses)
        return self._com
    
    @property
    def aromatic_atoms(self) -> List[Atom]:
        """Retourne les atomes du cycle aromatique (si présent)."""
        if self._aromatic_atoms is None:
            aromatic_names = ATOM_DEFINITIONS['aromatic_atoms']
            self._aromatic_atoms = [a for a in self.atoms if a.name in aromatic_names]
        return self._aromatic_atoms
    
    @property
    def aromatic_centroid(self) -> Optional[np.ndarray]:
        """
        Calcule le centroïde du cycle aromatique.
        Retourne None si l'analogue n'a pas de cycle aromatique.
        """
        if self._aromatic_com is None:
            arom = self.aromatic_atoms
            if len(arom) >= 5:  # Minimum pour un cycle aromatique
                self._aromatic_com = np.mean([a.coords for a in arom], axis=0)
        return self._aromatic_com
    
    @property
    def has_aromatic_ring(self) -> bool:
        """Vérifie si l'analogue possède un cycle aromatique."""
        return len(self.aromatic_atoms) >= 5
    
    def get_donors(self) -> List[Atom]:
        """Retourne les atomes donneurs de liaison H (N, O avec H)."""
        return [a for a in self.heavy_atoms if a.element in {'N', 'O'}]
    
    def get_acceptors(self) -> List[Atom]:
        """Retourne les atomes accepteurs de liaison H."""
        return [a for a in self.heavy_atoms if a.element in {'N', 'O', 'S'}]
    
    def get_hydrophobic_atoms(self) -> List[Atom]:
        """
        Retourne les atomes hydrophobes (carbones non polaires).
        Exclut les carbones du backbone et carbonyles.
        """
        polar_carbons = {'C', 'CA', 'C1', 'C2', 'C3'}  # Backbone/carbonyles typiques
        return [a for a in self.atoms 
                if a.element == 'C' and a.name not in polar_carbons]


# Masses atomiques standards (en u.m.a.)
ATOMIC_MASSES = {
    'H': 1.008,
    'C': 12.011,
    'N': 14.007,
    'O': 15.999,
    'S': 32.065,
    'P': 30.974,
}


@dataclass
class Frame:
    """
    Représentation d'une frame (snapshot) du système.
    
    Attributes
    ----------
    file_path : Path
        Chemin vers le fichier PDB
    system_name : str
        Nom du système (ex: scy, scf)
    analogues : List[Analogue]
        Liste des analogues dans cette frame
    box : Optional[np.ndarray]
        Dimensions de la boîte de simulation
    """
    file_path: Path
    system_name: str
    analogues: List[Analogue] = field(default_factory=list)
    box: Optional[np.ndarray] = None
    
    @property
    def n_analogues(self) -> int:
        """Nombre d'analogues dans la frame."""
        return len(self.analogues)
    
    def get_analogue_by_resid(self, resid: int) -> Optional[Analogue]:
        """Trouve un analogue par son numéro de résidu."""
        for ana in self.analogues:
            if ana.resid == resid:
                return ana
        return None


print("✓ Structures de données définies")

✓ Structures de données définies


### 3.2 Chargement des structures PDB

Fonctions pour lire les fichiers PDB et extraire les analogues avec leurs propriétés atomiques.

In [7]:
# =============================================================================
# Chargement des fichiers PDB avec MDAnalysis
# =============================================================================

def load_frame_mda(pdb_path: Path, system_name: str) -> Frame:
    """
    Charge une frame PDB et extrait les analogues avec MDAnalysis.
    
    Parameters
    ----------
    pdb_path : Path
        Chemin vers le fichier PDB
    system_name : str
        Nom du système (ex: scy)
        
    Returns
    -------
    Frame
        Objet Frame contenant les analogues
    """
    # Déterminer le nom de résidu attendu
    resname = ANALOGUE_INFO.get(system_name, (system_name.upper(),))[0]
    
    # Charger la structure avec MDAnalysis
    u = mda.Universe(str(pdb_path))
    
    # Extraire les dimensions de la boîte (si disponibles)
    box = u.dimensions[:3] if u.dimensions is not None else None
    
    # Sélectionner les analogues
    try:
        # Essayer avec le nom de résidu exact
        selection = u.select_atoms(f"resname {resname}")
    except:
        # Fallback: sélectionner tout ce qui n'est pas membrane/eau/ions
        selection = u.select_atoms("not (resname POPC* or resname TIP* or resname SOD or resname CLA or resname POT)")
    
    if len(selection) == 0:
        # Retourner une frame vide si aucun analogue trouvé
        return Frame(file_path=pdb_path, system_name=system_name, box=box)
    
    # Grouper par résidu
    residues = selection.residues
    analogues = []
    
    for res in residues:
        atoms_list = []
        for atom in res.atoms:
            # Déterminer l'élément à partir du nom d'atome
            element = atom.element if hasattr(atom, 'element') and atom.element else atom.name[0]
            
            atoms_list.append(Atom(
                name=atom.name,
                element=element,
                coords=atom.position.copy(),
                index=atom.index,
                mass=ATOMIC_MASSES.get(element, 12.0)
            ))
        
        analogues.append(Analogue(
            resid=res.resid,
            resname=res.resname,
            atoms=atoms_list
        ))
    
    return Frame(
        file_path=pdb_path,
        system_name=system_name,
        analogues=analogues,
        box=box
    )


def load_all_frames(systems: Dict[str, List[Path]], 
                    max_frames_per_system: Optional[int] = None,
                    verbose: bool = True) -> Dict[str, List[Frame]]:
    """
    Charge toutes les frames pour tous les systèmes.
    
    Parameters
    ----------
    systems : Dict[str, List[Path]]
        Dictionnaire {nom_système: [chemins_pdb]}
    max_frames_per_system : int, optional
        Limite le nombre de frames par système
    verbose : bool
        Affiche la progression
        
    Returns
    -------
    Dict[str, List[Frame]]
        Dictionnaire {nom_système: [frames]}
    """
    all_frames = {}
    
    for system_name, pdb_files in systems.items():
        if verbose:
            print(f"  Chargement de {system_name}...", end=" ")
        
        frames = []
        files_to_load = pdb_files[:max_frames_per_system] if max_frames_per_system else pdb_files
        
        for pdb_path in files_to_load:
            try:
                frame = load_frame_mda(pdb_path, system_name)
                if frame.n_analogues > 0:
                    frames.append(frame)
            except Exception as e:
                if verbose:
                    print(f"\n    ⚠ Erreur pour {pdb_path.name}: {e}")
        
        if frames:
            all_frames[system_name] = frames
            if verbose:
                print(f"✓ {len(frames)} frames, {frames[0].n_analogues} analogues/frame")
        else:
            if verbose:
                print("✗ Aucune frame valide")
    
    return all_frames


# Chargement de toutes les frames
print("Chargement des structures PDB...")
all_frames = load_all_frames(pdb_systems)

print(f"\n✓ Chargement terminé: {len(all_frames)} systèmes valides")

Chargement des structures PDB...
  Chargement de sca... ✓ 3 frames, 26 analogues/frame
  Chargement de scc... ✓ 3 frames, 26 analogues/frame
  Chargement de sccm... ✓ 3 frames, 26 analogues/frame
  Chargement de scd... ✓ 3 frames, 26 analogues/frame
  Chargement de scdn... ✓ 3 frames, 26 analogues/frame
  Chargement de sce... ✓ 3 frames, 26 analogues/frame
  Chargement de scen... ✓ 3 frames, 26 analogues/frame
  Chargement de scf... ✓ 3 frames, 26 analogues/frame
  Chargement de schd... ✓ 3 frames, 26 analogues/frame
  Chargement de sche... ✓ 3 frames, 26 analogues/frame
  Chargement de schp... ✓ 3 frames, 26 analogues/frame
  Chargement de sci... ✓ 3 frames, 26 analogues/frame
  Chargement de sck... ✓ 3 frames, 26 analogues/frame
  Chargement de sckn... ✓ 3 frames, 26 analogues/frame
  Chargement de scl... ✓ 3 frames, 26 analogues/frame
  Chargement de scm... ✓ 3 frames, 26 analogues/frame
  Chargement de scn... ✓ 3 frames, 26 analogues/frame
  Chargement de scp... ✓ 3 frames, 26 anal

## 4. Définitions des contacts et calculs de distances

### 4.1 Définition 1 : Distance minimale entre atomes lourds

La première définition du contact est basée sur la distance minimale entre paires d'atomes lourds appartenant à deux analogues distincts :

$$d_{\min}(A, B) = \min_{i \in A, j \in B} \|\mathbf{r}_i - \mathbf{r}_j\|$$

où $\mathbf{r}_i$ et $\mathbf{r}_j$ sont les positions des atomes lourds des analogues A et B.

In [ ]:
# =============================================================================
# Calcul de distances entre analogues
# =============================================================================

def min_heavy_atom_distance(ana1: Analogue, ana2: Analogue) -> float:
    """
    Calcule la distance minimale entre atomes lourds de deux analogues.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float
        Distance minimale en Ångströms
    """
    coords1 = ana1.heavy_coords
    coords2 = ana2.heavy_coords
    
    # Calcul efficace avec scipy
    dist_matrix = distance.cdist(coords1, coords2)
    return dist_matrix.min()


def com_distance(ana1: Analogue, ana2: Analogue) -> float:
    """
    Calcule la distance entre les centres de masse de deux analogues.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float
        Distance COM-COM en Ångströms
    """
    return np.linalg.norm(ana1.center_of_mass - ana2.center_of_mass)


def aromatic_centroid_distance(ana1: Analogue, ana2: Analogue) -> Optional[float]:
    """
    Calcule la distance entre les centroïdes des cycles aromatiques.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float or None
        Distance entre centroïdes, ou None si un des analogues n'a pas de cycle
    """
    c1 = ana1.aromatic_centroid
    c2 = ana2.aromatic_centroid
    
    if c1 is None or c2 is None:
        return None
    
    return np.linalg.norm(c1 - c2)


def compute_pairwise_distances(frame: Frame) -> Dict[str, np.ndarray]:
    """
    Calcule toutes les distances par paires d'analogues pour une frame.
    
    Parameters
    ----------
    frame : Frame
        La frame à analyser
        
    Returns
    -------
    Dict[str, np.ndarray]
        Dictionnaire contenant:
        - 'min_heavy': matrice des distances minimales atomes lourds
        - 'com': matrice des distances COM-COM
        - 'aromatic': matrice des distances centroïde-centroïde (NaN si non aromatique)
        - 'pairs': liste des paires (i, j) avec i < j
    """
    n = frame.n_analogues
    analogues = frame.analogues
    
    # Initialisation des matrices
    min_heavy = np.zeros((n, n))
    com_dist = np.zeros((n, n))
    aromatic_dist = np.full((n, n), np.nan)
    
    pairs = []
    
    for i in range(n):
        for j in range(i + 1, n):
            ana1, ana2 = analogues[i], analogues[j]
            
            # Distance minimale atomes lourds
            d_min = min_heavy_atom_distance(ana1, ana2)
            min_heavy[i, j] = min_heavy[j, i] = d_min
            
            # Distance COM
            d_com = com_distance(ana1, ana2)
            com_dist[i, j] = com_dist[j, i] = d_com
            
            # Distance aromatique (si applicable)
            d_arom = aromatic_centroid_distance(ana1, ana2)
            if d_arom is not None:
                aromatic_dist[i, j] = aromatic_dist[j, i] = d_arom
            
            pairs.append((i, j))
    
    return {
        'min_heavy': min_heavy,
        'com': com_dist,
        'aromatic': aromatic_dist,
        'pairs': pairs,
    }


# Test sur une frame
if all_frames:
    test_system = list(all_frames.keys())[0]
    test_frame = all_frames[test_system][0]
    
    print(f"Test de calcul des distances sur {test_system}...")
    distances = compute_pairwise_distances(test_frame)
    
    # Statistiques
    min_heavy_flat = distances['min_heavy'][np.triu_indices(test_frame.n_analogues, k=1)]
    com_flat = distances['com'][np.triu_indices(test_frame.n_analogues, k=1)]
    
    print(f"\n  Distance min atomes lourds:")
    print(f"    Min: {min_heavy_flat.min():.2f} Å, Max: {min_heavy_flat.max():.2f} Å, Moyenne: {min_heavy_flat.mean():.2f} Å")
    
    print(f"\n  Distance COM-COM:")
    print(f"    Min: {com_flat.min():.2f} Å, Max: {com_flat.max():.2f} Å, Moyenne: {com_flat.mean():.2f} Å")
    
    print("\n✓ Fonctions de calcul de distances opérationnelles")

Test de calcul des distances sur sca...


NameError: name 'spatial' is not defined

### 4.2 Définition 2 : Distance entre centres de masse avec cutoffs dépendant de l'analogue

La seconde définition utilise la distance entre centres de masse avec un cutoff adapté à la taille des analogues :

$$d_{\text{COM}}(A, B) = \|\mathbf{R}_{\text{COM}}^A - \mathbf{R}_{\text{COM}}^B\|$$

Le contact est établi si $d_{\text{COM}} < r_A + r_B + d_{\text{tol}}$, où $r_A$ et $r_B$ sont les rayons effectifs des analogues.

In [ ]:
# =============================================================================
# Fonctions de détection de contacts avec cutoff adaptatif
# =============================================================================

def compute_adaptive_cutoff(system_name: str, tolerance: float = 2.0) -> float:
    """
    Calcule un cutoff COM adapté au système.
    
    Le cutoff est basé sur la somme des rayons effectifs (2 * rayon) 
    plus une tolérance pour les contacts.
    
    Parameters
    ----------
    system_name : str
        Nom du système
    tolerance : float
        Tolérance additionnelle en Å
        
    Returns
    -------
    float
        Cutoff adaptatif en Å
    """
    radius = ANALOGUE_RADII.get(system_name, 3.5)
    return 2 * radius + tolerance


def detect_contacts_min_heavy(frame: Frame, cutoff: float) -> List[Tuple[int, int, float]]:
    """
    Détecte les contacts basés sur la distance minimale atomes lourds.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance en Å
        
    Returns
    -------
    List[Tuple[int, int, float]]
        Liste de (resid1, resid2, distance) pour les contacts
    """
    contacts = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = min_heavy_atom_distance(analogues[i], analogues[j])
            if dist <= cutoff:
                contacts.append((analogues[i].resid, analogues[j].resid, dist))
    
    return contacts


def detect_contacts_com(frame: Frame, cutoff: float) -> List[Tuple[int, int, float]]:
    """
    Détecte les contacts basés sur la distance COM-COM.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance en Å
        
    Returns
    -------
    List[Tuple[int, int, float]]
        Liste de (resid1, resid2, distance) pour les contacts
    """
    contacts = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = com_distance(analogues[i], analogues[j])
            if dist <= cutoff:
                contacts.append((analogues[i].resid, analogues[j].resid, dist))
    
    return contacts


def count_contacts_per_analogue(contacts: List[Tuple[int, int, float]], 
                                 n_analogues: int) -> np.ndarray:
    """
    Compte le nombre de contacts par analogue.
    
    Parameters
    ----------
    contacts : List[Tuple[int, int, float]]
        Liste des contacts détectés
    n_analogues : int
        Nombre total d'analogues
        
    Returns
    -------
    np.ndarray
        Nombre de contacts pour chaque analogue
    """
    counts = np.zeros(n_analogues, dtype=int)
    for resid1, resid2, _ in contacts:
        counts[resid1 - 1] += 1
        counts[resid2 - 1] += 1
    return counts


print("✓ Fonctions de détection de contacts définies")

## 5. Détection par types d'interactions

### 5.1 Interactions de van der Waals

Les interactions de van der Waals sont détectées sur la base de la distance entre atomes lourds. Un contact VdW est établi lorsque la distance inter-atomique est inférieure à la somme des rayons de van der Waals plus une tolérance :

$$d_{ij} \leq r_i^{\text{VdW}} + r_j^{\text{VdW}} + d_{\text{tol}}$$

Le cutoff typique est de 4.0 Å pour une détection générale.

In [ ]:
# =============================================================================
# Détection des interactions de van der Waals
# =============================================================================

# Rayons de van der Waals standards (en Å)
VDW_RADII = {
    'H': 1.20,
    'C': 1.70,
    'N': 1.55,
    'O': 1.52,
    'S': 1.80,
    'P': 1.80,
}


def detect_vdw_contacts(ana1: Analogue, ana2: Analogue, 
                        cutoff: float = 4.0) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les contacts de van der Waals entre deux analogues.
    
    Un contact VdW est établi si au moins une paire d'atomes lourds
    est à une distance inférieure au cutoff.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance en Å (défaut: 4.0)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_en_contact)
    """
    coords1 = ana1.heavy_coords
    coords2 = ana2.heavy_coords
    
    # Calcul de la matrice de distances
    dist_matrix = spatial.distance.cdist(coords1, coords2)
    
    # Distance minimale
    min_dist = dist_matrix.min()
    
    # Trouver toutes les paires en contact
    contact_pairs = []
    if min_dist <= cutoff:
        indices = np.where(dist_matrix <= cutoff)
        for idx1, idx2 in zip(indices[0], indices[1]):
            atom1 = ana1.heavy_atoms[idx1]
            atom2 = ana2.heavy_atoms[idx2]
            contact_pairs.append((atom1.name, atom2.name, dist_matrix[idx1, idx2]))
    
    return (min_dist <= cutoff, min_dist, contact_pairs)


def analyze_vdw_frame(frame: Frame, cutoff: float = 4.0) -> Dict:
    """
    Analyse complète des contacts VdW pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance VdW
        
    Returns
    -------
    Dict
        Résultats de l'analyse incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_vdw_contacts(
                analogues[i], analogues[j], cutoff
            )
            all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_atom_pairs': len(pairs),
                    'atom_pairs': pairs[:5],  # Garder les 5 premières paires
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'n_pairs_total': len(all_distances),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection VdW définie")

### 5.2 Liaisons hydrogène

Les liaisons hydrogène sont détectées entre atomes donneurs (N, O avec H attaché) et accepteurs (N, O, S avec paires d'électrons libres). Le critère géométrique standard est :

$$d_{\text{D-A}} \leq 3.5 \text{ Å}$$

où D est l'atome donneur et A l'accepteur. Pour une analyse plus rigoureuse, on peut aussi vérifier l'angle D-H...A.

In [ ]:
# =============================================================================
# Détection des liaisons hydrogène
# =============================================================================

def detect_hbond_contacts(ana1: Analogue, ana2: Analogue, 
                          cutoff: float = 3.5) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les liaisons hydrogène potentielles entre deux analogues.
    
    Basé sur la distance donneur-accepteur (D-A). Les atomes donneurs sont
    N et O pouvant porter un H. Les accepteurs sont N, O et S.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance D-A en Å (défaut: 3.5)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_H-bond)
    """
    donors1 = ana1.get_donors()
    acceptors1 = ana1.get_acceptors()
    donors2 = ana2.get_donors()
    acceptors2 = ana2.get_acceptors()
    
    if not (donors1 or donors2) or not (acceptors1 or acceptors2):
        return (False, np.inf, [])
    
    hbond_pairs = []
    min_dist = np.inf
    
    # Ana1 donneur -> Ana2 accepteur
    for d in donors1:
        for a in acceptors2:
            dist = d.distance_to(a)
            min_dist = min(min_dist, dist)
            if dist <= cutoff:
                hbond_pairs.append((d.name, a.name, dist))
    
    # Ana2 donneur -> Ana1 accepteur
    for d in donors2:
        for a in acceptors1:
            dist = d.distance_to(a)
            min_dist = min(min_dist, dist)
            if dist <= cutoff:
                hbond_pairs.append((d.name, a.name, dist))
    
    return (len(hbond_pairs) > 0, min_dist, hbond_pairs)


def analyze_hbond_frame(frame: Frame, cutoff: float = 3.5) -> Dict:
    """
    Analyse complète des liaisons H potentielles pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance D-A
        
    Returns
    -------
    Dict
        Résultats incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_hbond_contacts(
                analogues[i], analogues[j], cutoff
            )
            
            if min_dist < np.inf:
                all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_hbonds': len(pairs),
                    'hbond_pairs': pairs,
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection liaisons H définie")

### 5.3 Interactions hydrophobes

Les interactions hydrophobes sont détectées entre atomes de carbone aliphatiques (non polaires). Le critère est basé sur la distance C-C :

$$d_{\text{C-C}} \leq 4.5 \text{ Å}$$

Seuls les carbones non liés à des hétéroatomes (N, O, S) sont considérés comme hydrophobes.

In [ ]:
# =============================================================================
# Détection des interactions hydrophobes
# =============================================================================

def detect_hydrophobic_contacts(ana1: Analogue, ana2: Analogue, 
                                 cutoff: float = 4.5) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les contacts hydrophobes entre deux analogues.
    
    Basé sur les distances entre atomes de carbone (considérés hydrophobes).
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance C-C en Å (défaut: 4.5)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_hydrophobes)
    """
    hydro1 = ana1.get_hydrophobic_atoms()
    hydro2 = ana2.get_hydrophobic_atoms()
    
    if not hydro1 or not hydro2:
        return (False, np.inf, [])
    
    # Coordonnées des atomes hydrophobes
    coords1 = np.array([a.coords for a in hydro1])
    coords2 = np.array([a.coords for a in hydro2])
    
    # Matrice de distances
    dist_matrix = spatial.distance.cdist(coords1, coords2)
    min_dist = dist_matrix.min()
    
    # Paires en contact
    hydro_pairs = []
    if min_dist <= cutoff:
        indices = np.where(dist_matrix <= cutoff)
        for idx1, idx2 in zip(indices[0], indices[1]):
            hydro_pairs.append((
                hydro1[idx1].name, 
                hydro2[idx2].name, 
                dist_matrix[idx1, idx2]
            ))
    
    return (len(hydro_pairs) > 0, min_dist, hydro_pairs)


def analyze_hydrophobic_frame(frame: Frame, cutoff: float = 4.5) -> Dict:
    """
    Analyse complète des contacts hydrophobes pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance C-C
        
    Returns
    -------
    Dict
        Résultats incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_hydrophobic_contacts(
                analogues[i], analogues[j], cutoff
            )
            
            if min_dist < np.inf:
                all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_contacts': len(pairs),
                    'contact_pairs': pairs[:5],
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection contacts hydrophobes définie")

### 5.4 Interactions π–π (stacking aromatique)

Les interactions π–π sont détectées entre cycles aromatiques. Le critère principal est basé sur la distance entre les centroïdes des cycles :

$$d_{\text{centroid}} \leq 5.0 \text{ Å}$$

Cette analyse s'applique uniquement aux analogues aromatiques (Phe, Tyr, Trp, His).

In [ ]:
# =============================================================================
# Détection des interactions π–π
# =============================================================================

def compute_ring_normal(atoms: List[Atom]) -> Optional[np.ndarray]:
    """
    Calcule le vecteur normal au plan du cycle aromatique.
    
    Utilise les 3 premiers atomes pour définir le plan par produit vectoriel.
    
    Parameters
    ----------
    atoms : List[Atom]
        Atomes du cycle aromatique (minimum 3)
        
    Returns
    -------
    np.ndarray or None
        Vecteur normal unitaire, ou None si < 3 atomes
    """
    if len(atoms) < 3:
        return None
    
    # Vecteurs dans le plan du cycle
    v1 = atoms[1].coords - atoms[0].coords
    v2 = atoms[2].coords - atoms[0].coords
    
    # Produit vectoriel
    normal = np.cross(v1, v2)
    norm = np.linalg.norm(normal)
    
    if norm < 1e-6:
        return None
    
    return normal / norm


def compute_ring_angle(normal1: np.ndarray, normal2: np.ndarray) -> float:
    """
    Calcule l'angle entre deux plans aromatiques.
    
    Parameters
    ----------
    normal1, normal2 : np.ndarray
        Vecteurs normaux aux plans
        
    Returns
    -------
    float
        Angle en degrés (0° = parallèle, 90° = perpendiculaire)
    """
    cos_angle = abs(np.dot(normal1, normal2))
    cos_angle = min(1.0, max(-1.0, cos_angle))  # Clamp pour éviter erreurs numériques
    return np.degrees(np.arccos(cos_angle))


def detect_pipi_contacts(ana1: Analogue, ana2: Analogue, 
                         cutoff: float = 5.0) -> Tuple[bool, Optional[float], Optional[float]]:
    """
    Détecte les interactions π–π entre deux analogues aromatiques.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance centroïde-centroïde en Å (défaut: 5.0)
        
    Returns
    -------
    Tuple[bool, Optional[float], Optional[float]]
        (contact_detecté, distance_centroïdes, angle_entre_plans)
    """
    # Vérifier que les deux analogues ont des cycles aromatiques
    if not ana1.has_aromatic_ring or not ana2.has_aromatic_ring:
        return (False, None, None)
    
    # Distance entre centroïdes
    c1 = ana1.aromatic_centroid
    c2 = ana2.aromatic_centroid
    dist = np.linalg.norm(c1 - c2)
    
    # Angle entre les plans
    normal1 = compute_ring_normal(ana1.aromatic_atoms)
    normal2 = compute_ring_normal(ana2.aromatic_atoms)
    
    angle = None
    if normal1 is not None and normal2 is not None:
        angle = compute_ring_angle(normal1, normal2)
    
    return (dist <= cutoff, dist, angle)


def analyze_pipi_frame(frame: Frame, cutoff: float = 5.0) -> Dict:
    """
    Analyse complète des interactions π–π pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser (doit contenir des analogues aromatiques)
    cutoff : float
        Seuil de distance centroïde-centroïde
        
    Returns
    -------
    Dict
        Résultats incluant contacts, distances et angles
    """
    contacts = []
    all_distances = []
    all_angles = []
    
    # Filtrer les analogues aromatiques
    aromatic_analogues = [a for a in frame.analogues if a.has_aromatic_ring]
    n = len(aromatic_analogues)
    
    if n < 2:
        return {
            'contacts': [],
            'n_contacts': 0,
            'n_aromatic_pairs': 0,
            'distances': np.array([]),
            'angles': np.array([]),
        }
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, dist, angle = detect_pipi_contacts(
                aromatic_analogues[i], aromatic_analogues[j], cutoff
            )
            
            if dist is not None:
                all_distances.append(dist)
            if angle is not None:
                all_angles.append(angle)
            
            if has_contact:
                contacts.append({
                    'resid1': aromatic_analogues[i].resid,
                    'resid2': aromatic_analogues[j].resid,
                    'distance': dist,
                    'angle': angle,
                    'geometry': 'parallel' if angle < 30 else ('T-shaped' if angle > 60 else 'tilted'),
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'n_aromatic_pairs': len(all_distances),
        'distances': np.array(all_distances),
        'angles': np.array(all_angles),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection π–π définie")

## 6. Détection de contacts avec Biopython

Implémentation d'une détection de contacts basée sur les outils de Biopython (`NeighborSearch`) pour permettre une comparaison directe avec les méthodes précédentes.

In [ ]:
# =============================================================================
# Détection de contacts avec Biopython (NeighborSearch)
# =============================================================================

def load_pdb_biopython(pdb_path: Path, system_name: str) -> Tuple[List, Dict]:
    """
    Charge un fichier PDB avec Biopython et extrait les atomes des analogues.
    
    Parameters
    ----------
    pdb_path : Path
        Chemin vers le fichier PDB
    system_name : str
        Nom du système
        
    Returns
    -------
    Tuple[List, Dict]
        (liste_atomes, dict_residus)
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('system', str(pdb_path))
    
    resname = ANALOGUE_INFO.get(system_name, (system_name.upper(),))[0]
    
    atoms = []
    residues = {}
    
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.get_resname() == resname:
                    resid = residue.get_id()[1]
                    residues[resid] = residue
                    for atom in residue:
                        # Exclure les hydrogènes
                        if atom.element != 'H':
                            atoms.append(atom)
    
    return atoms, residues


def detect_contacts_biopython(pdb_path: Path, system_name: str, 
                               cutoff: float = 4.0) -> Dict:
    """
    Détecte les contacts entre analogues en utilisant Biopython NeighborSearch.
    
    Cette méthode utilise un arbre KD pour une recherche efficace des voisins.
    
    Parameters
    ----------
    pdb_path : Path
        Chemin vers le fichier PDB
    system_name : str
        Nom du système
    cutoff : float
        Seuil de distance en Å
        
    Returns
    -------
    Dict
        Résultats de l'analyse avec contacts et statistiques
    """
    atoms, residues = load_pdb_biopython(pdb_path, system_name)
    
    if not atoms:
        return {'contacts': [], 'n_contacts': 0, 'n_residues': 0}
    
    # Création du NeighborSearch avec un arbre KD
    ns = NeighborSearch(atoms)
    
    # Recherche des contacts
    contacts_set = set()
    contact_details = []
    
    for atom in atoms:
        # Trouver tous les atomes dans le cutoff
        neighbors = ns.search(atom.coord, cutoff, level='A')
        
        for neighbor in neighbors:
            if neighbor == atom:
                continue
            
            # Récupérer les résidus parents
            res1 = atom.get_parent()
            res2 = neighbor.get_parent()
            resid1 = res1.get_id()[1]
            resid2 = res2.get_id()[1]
            
            # Éviter les contacts intra-résidu et les doublons
            if resid1 != resid2:
                pair = tuple(sorted([resid1, resid2]))
                if pair not in contacts_set:
                    contacts_set.add(pair)
                    dist = atom - neighbor  # Distance entre atomes
                    contact_details.append({
                        'resid1': pair[0],
                        'resid2': pair[1],
                        'atom1': atom.get_name(),
                        'atom2': neighbor.get_name(),
                        'distance': dist,
                    })
    
    return {
        'contacts': contact_details,
        'n_contacts': len(contacts_set),
        'contact_pairs': list(contacts_set),
        'n_residues': len(residues),
    }


def analyze_system_biopython(system_name: str, pdb_files: List[Path], 
                              cutoff: float = 4.0) -> Dict:
    """
    Analyse tous les fichiers PDB d'un système avec Biopython.
    
    Parameters
    ----------
    system_name : str
        Nom du système
    pdb_files : List[Path]
        Liste des fichiers PDB
    cutoff : float
        Seuil de distance
        
    Returns
    -------
    Dict
        Résultats agrégés sur toutes les frames
    """
    all_contacts = []
    all_n_contacts = []
    
    for pdb_file in pdb_files:
        try:
            result = detect_contacts_biopython(pdb_file, system_name, cutoff)
            all_contacts.extend(result['contacts'])
            all_n_contacts.append(result['n_contacts'])
        except Exception as e:
            print(f"  ⚠ Erreur Biopython pour {pdb_file.name}: {e}")
    
    return {
        'system': system_name,
        'n_frames': len(pdb_files),
        'total_contacts': len(all_contacts),
        'contacts_per_frame': np.array(all_n_contacts),
        'mean_contacts': np.mean(all_n_contacts) if all_n_contacts else 0,
        'std_contacts': np.std(all_n_contacts) if all_n_contacts else 0,
    }


print("✓ Détection Biopython définie")

## 7. Fonction de Distribution Radiale (RDF)

### 7.1 Calcul de la RDF entre analogues

La fonction de distribution radiale $g(r)$ caractérise la probabilité de trouver un analogue à une distance $r$ d'un autre analogue, normalisée par la densité moyenne. Pour des frames indépendantes (non une trajectoire continue), on agrège les histogrammes de distances sur toutes les frames.

La formule utilisée est compatible avec les calculs NAMD :

$$g(r) = \frac{\langle n(r) \rangle}{4\pi r^2 \Delta r \cdot \rho}$$

où $n(r)$ est le nombre de paires dans la coquille $[r, r+\Delta r]$ et $\rho$ est la densité numérique.

In [ ]:
# =============================================================================
# Calcul de la fonction de distribution radiale (RDF)
# =============================================================================

def compute_rdf_frame(frame: Frame, 
                      r_max: float = 30.0, 
                      dr: float = 0.1,
                      distance_type: str = 'com') -> Tuple[np.ndarray, np.ndarray]:
    """
    Calcule l'histogramme des distances pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    r_max : float
        Distance maximale en Å
    dr : float
        Largeur des bins en Å
    distance_type : str
        'com' pour centre de masse, 'min_heavy' pour distance minimale
        
    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        (centres_bins, comptages)
    """
    n = frame.n_analogues
    distances = []
    
    for i in range(n):
        for j in range(i + 1, n):
            if distance_type == 'com':
                d = com_distance(frame.analogues[i], frame.analogues[j])
            else:
                d = min_heavy_atom_distance(frame.analogues[i], frame.analogues[j])
            distances.append(d)
    
    # Histogramme
    bins = np.arange(0, r_max + dr, dr)
    counts, _ = np.histogram(distances, bins=bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    return bin_centers, counts


def compute_rdf_system(frames: List[Frame], 
                       r_max: float = 30.0, 
                       dr: float = 0.1,
                       distance_type: str = 'com',
                       normalize: bool = True) -> Dict:
    """
    Calcule la RDF agrégée sur toutes les frames d'un système.
    
    Pour des frames indépendantes, on somme les histogrammes puis on normalise.
    La normalisation utilise la formule standard de la RDF :
    g(r) = n(r) / (4*pi*r^2*dr * rho * N_frames * N_pairs)
    
    Parameters
    ----------
    frames : List[Frame]
        Liste des frames à analyser
    r_max : float
        Distance maximale en Å
    dr : float
        Largeur des bins en Å (compatible NAMD: 0.1 Å typique)
    distance_type : str
        'com' ou 'min_heavy'
    normalize : bool
        Si True, normalise en g(r); sinon retourne les comptages bruts
        
    Returns
    -------
    Dict
        Dictionnaire avec r, g(r), erreurs, et métadonnées
    """
    bins = np.arange(0, r_max + dr, dr)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    # Accumuler les histogrammes de toutes les frames
    total_counts = np.zeros(len(bin_centers))
    counts_per_frame = []
    
    n_pairs_total = 0
    
    for frame in frames:
        _, counts = compute_rdf_frame(frame, r_max, dr, distance_type)
        total_counts += counts
        counts_per_frame.append(counts)
        
        # Nombre de paires dans cette frame
        n = frame.n_analogues
        n_pairs_total += n * (n - 1) // 2
    
    counts_per_frame = np.array(counts_per_frame)
    n_frames = len(frames)
    
    if not normalize:
        return {
            'r': bin_centers,
            'counts': total_counts,
            'counts_per_frame': counts_per_frame,
            'n_frames': n_frames,
        }
    
    # Normalisation pour obtenir g(r)
    # Volume de la coquille sphérique à distance r
    shell_volumes = 4 * np.pi * bin_centers**2 * dr
    
    # Densité moyenne (approximation: utilise les dimensions de boîte si disponibles)
    # Pour frames indépendantes, on calcule une densité effective
    if frames[0].box is not None:
        box = frames[0].box
        volume = box[0] * box[1] * box[2]
        n_analogues = frames[0].n_analogues
        rho = n_analogues / volume
    else:
        # Estimation basée sur la distribution des distances
        rho = 1.0 / (4/3 * np.pi * (r_max/2)**3)  # Approximation
    
    # g(r) = <n(r)> / (shell_volume * rho * N_reference)
    # Où N_reference = nombre d'analogues de référence (tous les analogues)
    n_ref = frames[0].n_analogues * n_frames
    
    # Éviter division par zéro pour r proche de 0
    with np.errstate(divide='ignore', invalid='ignore'):
        g_r = total_counts / (shell_volumes * rho * n_ref)
        g_r = np.nan_to_num(g_r, nan=0.0, posinf=0.0)
    
    # Calcul de l'erreur (écart-type sur les frames)
    if n_frames > 1:
        # Normaliser chaque frame individuellement
        g_r_per_frame = counts_per_frame / (shell_volumes * rho * frames[0].n_analogues)
        g_r_std = np.std(g_r_per_frame, axis=0)
        g_r_sem = g_r_std / np.sqrt(n_frames)
    else:
        g_r_std = np.zeros_like(g_r)
        g_r_sem = np.zeros_like(g_r)
    
    return {
        'r': bin_centers,
        'g_r': g_r,
        'g_r_std': g_r_std,
        'g_r_sem': g_r_sem,
        'counts': total_counts,
        'n_frames': n_frames,
        'n_pairs_total': n_pairs_total,
        'rho': rho,
        'dr': dr,
        'distance_type': distance_type,
    }


def compute_coordination_number(rdf_result: Dict, r_cutoff: float) -> float:
    """
    Calcule le nombre de coordination à partir de la RDF.
    
    Le nombre de coordination N(r) est l'intégrale de 4*pi*r^2*rho*g(r) de 0 à r_cutoff.
    
    Parameters
    ----------
    rdf_result : Dict
        Résultat de compute_rdf_system
    r_cutoff : float
        Distance de cutoff pour l'intégration
        
    Returns
    -------
    float
        Nombre de coordination moyen
    """
    r = rdf_result['r']
    g_r = rdf_result['g_r']
    rho = rdf_result['rho']
    dr = rdf_result['dr']
    
    # Masque pour r <= r_cutoff
    mask = r <= r_cutoff
    
    # Intégration numérique (méthode des trapèzes simplifiée)
    integrand = 4 * np.pi * r[mask]**2 * rho * g_r[mask] * dr
    
    return np.sum(integrand)


print("✓ Calcul RDF défini")

## 8. Analyse paramétrique : Balayage des cutoffs

### 8.1 Balayage pour la méthode distance minimale atomes lourds

Exploration systématique des cutoffs de 2.5 Å à 8.0 Å pour quantifier l'évolution du nombre de contacts détectés.

In [ ]:
# =============================================================================
# Balayage des cutoffs - Distance minimale atomes lourds
# =============================================================================

def cutoff_sweep_min_heavy(frames: List[Frame], 
                            cutoffs: np.ndarray) -> pd.DataFrame:
    """
    Effectue un balayage de cutoffs pour la méthode distance minimale.
    
    Parameters
    ----------
    frames : List[Frame]
        Frames à analyser
    cutoffs : np.ndarray
        Valeurs de cutoff à tester
        
    Returns
    -------
    pd.DataFrame
        Tableau avec nombre de contacts pour chaque cutoff et frame
    """
    results = []
    
    # Pré-calculer toutes les distances min pour chaque frame
    all_distances = []
    for frame in frames:
        dists = compute_pairwise_distances(frame)
        # Extraire distances uniques (triangle supérieur)
        n = frame.n_analogues
        min_heavy_flat = dists['min_heavy'][np.triu_indices(n, k=1)]
        all_distances.append(min_heavy_flat)
    
    # Pour chaque cutoff, compter les contacts
    for cutoff in cutoffs:
        for frame_idx, distances in enumerate(all_distances):
            n_contacts = np.sum(distances <= cutoff)
            n_pairs = len(distances)
            
            results.append({
                'cutoff': cutoff,
                'frame': frame_idx,
                'n_contacts': n_contacts,
                'n_pairs': n_pairs,
                'fraction': n_contacts / n_pairs if n_pairs > 0 else 0,
            })
    
    return pd.DataFrame(results)


def cutoff_sweep_com(frames: List[Frame], 
                      cutoffs: np.ndarray) -> pd.DataFrame:
    """
    Effectue un balayage de cutoffs pour la méthode COM.
    
    Parameters
    ----------
    frames : List[Frame]
        Frames à analyser
    cutoffs : np.ndarray
        Valeurs de cutoff à tester
        
    Returns
    -------
    pd.DataFrame
        Tableau avec nombre de contacts pour chaque cutoff et frame
    """
    results = []
    
    # Pré-calculer toutes les distances COM pour chaque frame
    all_distances = []
    for frame in frames:
        dists = compute_pairwise_distances(frame)
        n = frame.n_analogues
        com_flat = dists['com'][np.triu_indices(n, k=1)]
        all_distances.append(com_flat)
    
    # Pour chaque cutoff, compter les contacts
    for cutoff in cutoffs:
        for frame_idx, distances in enumerate(all_distances):
            n_contacts = np.sum(distances <= cutoff)
            n_pairs = len(distances)
            
            results.append({
                'cutoff': cutoff,
                'frame': frame_idx,
                'n_contacts': n_contacts,
                'n_pairs': n_pairs,
                'fraction': n_contacts / n_pairs if n_pairs > 0 else 0,
            })
    
    return pd.DataFrame(results)


def cutoff_sweep_by_interaction_type(frames: List[Frame], 
                                      cutoffs: np.ndarray) -> Dict[str, pd.DataFrame]:
    """
    Effectue un balayage de cutoffs pour chaque type d'interaction.
    
    Parameters
    ----------
    frames : List[Frame]
        Frames à analyser
    cutoffs : np.ndarray
        Valeurs de cutoff à tester
        
    Returns
    -------
    Dict[str, pd.DataFrame]
        DataFrames pour chaque type d'interaction
    """
    results = {
        'vdw': [],
        'hbond': [],
        'hydrophobic': [],
        'pipi': [],
    }
    
    for frame_idx, frame in enumerate(frames):
        analogues = frame.analogues
        n = len(analogues)
        
        # Pré-calculer toutes les distances par type
        vdw_dists = []
        hbond_dists = []
        hydro_dists = []
        pipi_dists = []
        
        for i in range(n):
            for j in range(i + 1, n):
                # VdW (distance min atomes lourds)
                _, d_vdw, _ = detect_vdw_contacts(analogues[i], analogues[j], cutoff=100)
                vdw_dists.append(d_vdw)
                
                # H-bond (distance D-A)
                _, d_hb, _ = detect_hbond_contacts(analogues[i], analogues[j], cutoff=100)
                if d_hb < np.inf:
                    hbond_dists.append(d_hb)
                
                # Hydrophobe
                _, d_hy, _ = detect_hydrophobic_contacts(analogues[i], analogues[j], cutoff=100)
                if d_hy < np.inf:
                    hydro_dists.append(d_hy)
                
                # π-π
                _, d_pi, _ = detect_pipi_contacts(analogues[i], analogues[j], cutoff=100)
                if d_pi is not None:
                    pipi_dists.append(d_pi)
        
        # Compter les contacts pour chaque cutoff
        for cutoff in cutoffs:
            results['vdw'].append({
                'cutoff': cutoff, 'frame': frame_idx,
                'n_contacts': sum(1 for d in vdw_dists if d <= cutoff),
                'n_pairs': len(vdw_dists),
            })
            results['hbond'].append({
                'cutoff': cutoff, 'frame': frame_idx,
                'n_contacts': sum(1 for d in hbond_dists if d <= cutoff),
                'n_pairs': len(hbond_dists),
            })
            results['hydrophobic'].append({
                'cutoff': cutoff, 'frame': frame_idx,
                'n_contacts': sum(1 for d in hydro_dists if d <= cutoff),
                'n_pairs': len(hydro_dists),
            })
            results['pipi'].append({
                'cutoff': cutoff, 'frame': frame_idx,
                'n_contacts': sum(1 for d in pipi_dists if d <= cutoff),
                'n_pairs': len(pipi_dists),
            })
    
    return {k: pd.DataFrame(v) for k, v in results.items()}


print("✓ Fonctions de balayage de cutoffs définies")

## 9. Exécution de l'analyse complète

### 9.1 Analyse pour tous les systèmes disponibles

Exécution systématique de l'analyse sur tous les systèmes d'analogues détectés.

In [ ]:
# =============================================================================
# Exécution de l'analyse complète
# =============================================================================

# Stockage des résultats
all_results = {}

print("=" * 70)
print("ANALYSE COMPLÈTE DES INTERACTIONS ENTRE ANALOGUES")
print("=" * 70)

for system_name, frames in all_frames.items():
    print(f"\n{'─' * 50}")
    print(f"Système: {system_name.upper()}")
    print(f"{'─' * 50}")
    
    info = ANALOGUE_INFO.get(system_name, (system_name.upper(), 'Unknown', 'unknown'))
    print(f"  Acide aminé: {info[1]} ({info[2]})")
    print(f"  Nombre de frames: {len(frames)}")
    print(f"  Analogues par frame: {frames[0].n_analogues}")
    
    # Initialiser le dictionnaire de résultats pour ce système
    system_results = {
        'info': info,
        'n_frames': len(frames),
        'n_analogues': frames[0].n_analogues,
    }
    
    # ---------------------------------------------------------------------
    # 1. Balayage des cutoffs - Distance minimale atomes lourds
    # ---------------------------------------------------------------------
    print("\n  [1/5] Balayage cutoffs (distance min atomes lourds)...", end=" ")
    cutoffs_min_heavy = CUTOFF_RANGES['min_heavy']
    df_min_heavy = cutoff_sweep_min_heavy(frames, cutoffs_min_heavy)
    system_results['cutoff_sweep_min_heavy'] = df_min_heavy
    print("✓")
    
    # ---------------------------------------------------------------------
    # 2. Balayage des cutoffs - Distance COM
    # ---------------------------------------------------------------------
    print("  [2/5] Balayage cutoffs (distance COM)...", end=" ")
    cutoffs_com = CUTOFF_RANGES['com']
    df_com = cutoff_sweep_com(frames, cutoffs_com)
    system_results['cutoff_sweep_com'] = df_com
    print("✓")
    
    # ---------------------------------------------------------------------
    # 3. Analyse par type d'interaction (avec cutoffs par défaut)
    # ---------------------------------------------------------------------
    print("  [3/5] Analyse par type d'interaction...", end=" ")
    
    interaction_results = {
        'vdw': [],
        'hbond': [],
        'hydrophobic': [],
        'pipi': [],
    }
    
    for frame in frames:
        vdw_res = analyze_vdw_frame(frame, DEFAULT_CUTOFFS['vdw'])
        hbond_res = analyze_hbond_frame(frame, DEFAULT_CUTOFFS['hbond'])
        hydro_res = analyze_hydrophobic_frame(frame, DEFAULT_CUTOFFS['hydrophobic'])
        pipi_res = analyze_pipi_frame(frame, DEFAULT_CUTOFFS['pipi'])
        
        interaction_results['vdw'].append(vdw_res)
        interaction_results['hbond'].append(hbond_res)
        interaction_results['hydrophobic'].append(hydro_res)
        interaction_results['pipi'].append(pipi_res)
    
    system_results['interactions'] = interaction_results
    print("✓")
    
    # Résumé des interactions
    for itype, results_list in interaction_results.items():
        n_contacts_mean = np.mean([r['n_contacts'] for r in results_list])
        frac_mean = np.mean([r.get('fraction_in_contact', 0) for r in results_list])
        print(f"    {itype.upper():12s}: {n_contacts_mean:.1f} contacts/frame ({frac_mean*100:.1f}%)")
    
    # ---------------------------------------------------------------------
    # 4. Analyse Biopython
    # ---------------------------------------------------------------------
    print("  [4/5] Analyse Biopython...", end=" ")
    try:
        pdb_files = pdb_systems[system_name]
        bp_results = analyze_system_biopython(system_name, pdb_files, cutoff=4.0)
        system_results['biopython'] = bp_results
        print(f"✓ ({bp_results['mean_contacts']:.1f} contacts/frame)")
    except Exception as e:
        print(f"⚠ Erreur: {e}")
        system_results['biopython'] = None
    
    # ---------------------------------------------------------------------
    # 5. Calcul de la RDF
    # ---------------------------------------------------------------------
    print("  [5/5] Calcul RDF...", end=" ")
    rdf_com = compute_rdf_system(frames, r_max=30.0, dr=0.1, distance_type='com')
    rdf_min = compute_rdf_system(frames, r_max=30.0, dr=0.1, distance_type='min_heavy')
    system_results['rdf_com'] = rdf_com
    system_results['rdf_min_heavy'] = rdf_min
    print("✓")
    
    # Sauvegarder les résultats
    all_results[system_name] = system_results

print("\n" + "=" * 70)
print(f"ANALYSE TERMINÉE: {len(all_results)} systèmes analysés")
print("=" * 70)

## 10. Visualisations

### 10.1 Distribution des distances inter-analogues

Histogrammes des distances minimales et COM pour chaque système.

In [ ]:
# =============================================================================
# Visualisation: Distribution des distances
# =============================================================================

def plot_distance_distributions(all_results: Dict, save_path: Optional[Path] = None):
    """
    Trace les distributions de distances pour tous les systèmes.
    """
    n_systems = len(all_results)
    if n_systems == 0:
        print("Aucun système à afficher")
        return
    
    # Disposition des subplots
    n_cols = min(4, n_systems)
    n_rows = (n_systems + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows))
    if n_systems == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (system_name, results) in enumerate(all_results.items()):
        row, col = idx // n_cols, idx % n_cols
        ax = axes[row, col]
        
        # Extraire les distances de la RDF (non normalisées)
        r = results['rdf_com']['r']
        counts_com = results['rdf_com']['counts']
        counts_min = results['rdf_min_heavy']['counts']
        
        # Normaliser pour comparaison
        if counts_com.sum() > 0:
            counts_com_norm = counts_com / counts_com.sum()
        else:
            counts_com_norm = counts_com
        if counts_min.sum() > 0:
            counts_min_norm = counts_min / counts_min.sum()
        else:
            counts_min_norm = counts_min
        
        # Tracer
        ax.plot(r, counts_min_norm, label='Min heavy atoms', color=COLORS['vdw'], alpha=0.8)
        ax.plot(r, counts_com_norm, label='COM-COM', color=COLORS['pipi'], alpha=0.8)
        
        # Lignes verticales pour les cutoffs par défaut
        ax.axvline(DEFAULT_CUTOFFS['vdw'], color=COLORS['vdw'], linestyle='--', alpha=0.5, lw=1)
        ax.axvline(compute_adaptive_cutoff(system_name), color=COLORS['pipi'], linestyle='--', alpha=0.5, lw=1)
        
        info = results['info']
        ax.set_title(f"{info[1]}\n({system_name.upper()})", fontsize=11)
        ax.set_xlabel('Distance (Å)')
        ax.set_ylabel('Fréquence normalisée')
        ax.set_xlim(0, 25)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # Masquer les axes vides
    for idx in range(n_systems, n_rows * n_cols):
        row, col = idx // n_cols, idx % n_cols
        axes[row, col].axis('off')
    
    plt.suptitle('Distribution des distances inter-analogues', fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_distance_distributions(all_results, FIGURES_DIR / 'distance_distributions.png')

### 10.2 Courbes de balayage des cutoffs

Évolution du nombre de contacts en fonction du cutoff pour les deux définitions du contact.

In [ ]:
# =============================================================================
# Visualisation: Balayage des cutoffs
# =============================================================================

def plot_cutoff_sweep(all_results: Dict, save_path: Optional[Path] = None):
    """
    Trace l'évolution du nombre de contacts vs cutoff pour tous les systèmes.
    """
    n_systems = len(all_results)
    if n_systems == 0:
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Palette de couleurs
    colors = plt.cm.tab20(np.linspace(0, 1, n_systems))
    
    # Panel A: Distance minimale atomes lourds
    ax1 = axes[0]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_min_heavy']
        # Moyenne par cutoff
        mean_contacts = df.groupby('cutoff')['n_contacts'].mean()
        std_contacts = df.groupby('cutoff')['n_contacts'].std()
        
        ax1.plot(mean_contacts.index, mean_contacts.values, 
                 label=system_name.upper(), color=colors[idx], lw=2)
        ax1.fill_between(mean_contacts.index, 
                         mean_contacts.values - std_contacts.values,
                         mean_contacts.values + std_contacts.values,
                         color=colors[idx], alpha=0.1)
    
    # Ligne verticale pour cutoff VdW standard
    ax1.axvline(DEFAULT_CUTOFFS['vdw'], color='red', linestyle='--', 
                label=f"Cutoff VdW ({DEFAULT_CUTOFFS['vdw']} Å)", lw=2)
    
    ax1.set_xlabel('Cutoff (Å)', fontsize=12)
    ax1.set_ylabel('Nombre de contacts', fontsize=12)
    ax1.set_title('A) Méthode: Distance minimale atomes lourds', fontsize=13)
    ax1.legend(fontsize=8, ncol=2, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(2.5, 8)
    
    # Panel B: Distance COM
    ax2 = axes[1]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_com']
        mean_contacts = df.groupby('cutoff')['n_contacts'].mean()
        std_contacts = df.groupby('cutoff')['n_contacts'].std()
        
        ax2.plot(mean_contacts.index, mean_contacts.values,
                 label=system_name.upper(), color=colors[idx], lw=2)
        ax2.fill_between(mean_contacts.index,
                         mean_contacts.values - std_contacts.values,
                         mean_contacts.values + std_contacts.values,
                         color=colors[idx], alpha=0.1)
    
    ax2.set_xlabel('Cutoff (Å)', fontsize=12)
    ax2.set_ylabel('Nombre de contacts', fontsize=12)
    ax2.set_title('B) Méthode: Distance centre de masse', fontsize=13)
    ax2.legend(fontsize=8, ncol=2, loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(4, 15)
    
    plt.suptitle('Balayage des cutoffs: Nombre de contacts vs. seuil de distance', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_cutoff_sweep(all_results, FIGURES_DIR / 'cutoff_sweep.png')

### 10.3 Fraction de paires en contact vs cutoff

Visualisation de la fraction de paires d'analogues en contact pour différents cutoffs.

In [ ]:
# =============================================================================
# Visualisation: Fraction de contacts vs cutoff
# =============================================================================

def plot_contact_fraction(all_results: Dict, save_path: Optional[Path] = None):
    """
    Trace la fraction de paires en contact vs cutoff.
    """
    n_systems = len(all_results)
    if n_systems == 0:
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab20(np.linspace(0, 1, n_systems))
    
    # Panel A: Distance minimale atomes lourds
    ax1 = axes[0]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_min_heavy']
        df['fraction'] = df['n_contacts'] / df['n_pairs']
        mean_frac = df.groupby('cutoff')['fraction'].mean() * 100
        
        ax1.plot(mean_frac.index, mean_frac.values,
                 label=system_name.upper(), color=colors[idx], lw=2)
    
    ax1.axvline(DEFAULT_CUTOFFS['vdw'], color='red', linestyle='--', lw=2)
    ax1.axhline(5, color='gray', linestyle=':', alpha=0.5, label='5% threshold')
    
    ax1.set_xlabel('Cutoff (Å)', fontsize=12)
    ax1.set_ylabel('Fraction de paires en contact (%)', fontsize=12)
    ax1.set_title('A) Méthode: Distance minimale atomes lourds', fontsize=13)
    ax1.legend(fontsize=8, ncol=2, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(2.5, 8)
    ax1.set_ylim(0, 50)
    
    # Panel B: Distance COM
    ax2 = axes[1]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_com']
        df['fraction'] = df['n_contacts'] / df['n_pairs']
        mean_frac = df.groupby('cutoff')['fraction'].mean() * 100
        
        ax2.plot(mean_frac.index, mean_frac.values,
                 label=system_name.upper(), color=colors[idx], lw=2)
    
    ax2.axhline(5, color='gray', linestyle=':', alpha=0.5, label='5% threshold')
    
    ax2.set_xlabel('Cutoff (Å)', fontsize=12)
    ax2.set_ylabel('Fraction de paires en contact (%)', fontsize=12)
    ax2.set_title('B) Méthode: Distance centre de masse', fontsize=13)
    ax2.legend(fontsize=8, ncol=2, loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(4, 15)
    ax2.set_ylim(0, 50)
    
    plt.suptitle('Fraction de paires d\'analogues en contact vs. cutoff', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_contact_fraction(all_results, FIGURES_DIR / 'contact_fraction.png')

### 10.4 Comparaison des types d'interactions

Comparaison du nombre de contacts détectés pour chaque type d'interaction (VdW, H-bond, hydrophobe, π-π).

In [ ]:
# =============================================================================
# Visualisation: Comparaison par type d'interaction
# =============================================================================

def plot_interaction_comparison(all_results: Dict, save_path: Optional[Path] = None):
    """
    Compare le nombre de contacts par type d'interaction pour chaque système.
    """
    # Préparer les données
    data = []
    for system_name, results in all_results.items():
        interactions = results['interactions']
        info = results['info']
        
        for itype in ['vdw', 'hbond', 'hydrophobic', 'pipi']:
            n_contacts = np.mean([r['n_contacts'] for r in interactions[itype]])
            n_std = np.std([r['n_contacts'] for r in interactions[itype]])
            data.append({
                'system': system_name.upper(),
                'aa_name': info[1],
                'aa_type': info[2],
                'interaction': itype,
                'n_contacts': n_contacts,
                'n_std': n_std,
            })
    
    df = pd.DataFrame(data)
    
    # Graphique en barres groupées
    fig, ax = plt.subplots(figsize=(14, 6))
    
    systems = df['system'].unique()
    n_systems = len(systems)
    n_types = 4
    width = 0.2
    x = np.arange(n_systems)
    
    for i, itype in enumerate(['vdw', 'hbond', 'hydrophobic', 'pipi']):
        subset = df[df['interaction'] == itype]
        bars = ax.bar(x + i*width, subset['n_contacts'], width,
                      label=itype.upper(), color=COLORS[itype], alpha=0.8,
                      yerr=subset['n_std'], capsize=3)
    
    ax.set_xlabel('Système', fontsize=12)
    ax.set_ylabel('Nombre moyen de contacts par frame', fontsize=12)
    ax.set_title('Comparaison des types d\'interactions par système\n'
                 f'(Cutoffs: VdW={DEFAULT_CUTOFFS["vdw"]}Å, H-bond={DEFAULT_CUTOFFS["hbond"]}Å, '
                 f'Hydrophobe={DEFAULT_CUTOFFS["hydrophobic"]}Å, π-π={DEFAULT_CUTOFFS["pipi"]}Å)',
                 fontsize=13)
    ax.set_xticks(x + 1.5*width)
    ax.set_xticklabels(systems, rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()
    
    return df


# Générer le graphique
if all_results:
    interaction_df = plot_interaction_comparison(all_results, FIGURES_DIR / 'interaction_comparison.png')

### 10.5 Fonctions de distribution radiale (RDF)

Visualisation des RDF calculées pour les deux méthodes de distance.

In [ ]:
# =============================================================================
# Visualisation: Fonctions de distribution radiale (RDF)
# =============================================================================

def plot_rdf(all_results: Dict, save_path: Optional[Path] = None):
    """
    Trace les RDF pour tous les systèmes.
    """
    n_systems = len(all_results)
    if n_systems == 0:
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab20(np.linspace(0, 1, n_systems))
    
    # Panel A: RDF basée sur distance min atomes lourds
    ax1 = axes[0]
    for idx, (system_name, results) in enumerate(all_results.items()):
        rdf = results['rdf_min_heavy']
        r = rdf['r']
        g_r = rdf['g_r']
        
        # Lissage pour meilleure visualisation
        g_r_smooth = uniform_filter1d(g_r, size=5)
        
        ax1.plot(r, g_r_smooth, label=system_name.upper(), color=colors[idx], lw=1.5)
    
    ax1.axhline(1, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(DEFAULT_CUTOFFS['vdw'], color='red', linestyle='--', 
                alpha=0.7, label=f"Cutoff VdW ({DEFAULT_CUTOFFS['vdw']} Å)")
    
    ax1.set_xlabel('r (Å)', fontsize=12)
    ax1.set_ylabel('g(r)', fontsize=12)
    ax1.set_title('A) RDF - Distance minimale atomes lourds', fontsize=13)
    ax1.legend(fontsize=8, ncol=2, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 20)
    ax1.set_ylim(0, None)
    
    # Panel B: RDF basée sur distance COM
    ax2 = axes[1]
    for idx, (system_name, results) in enumerate(all_results.items()):
        rdf = results['rdf_com']
        r = rdf['r']
        g_r = rdf['g_r']
        
        g_r_smooth = uniform_filter1d(g_r, size=5)
        
        ax2.plot(r, g_r_smooth, label=system_name.upper(), color=colors[idx], lw=1.5)
    
    ax2.axhline(1, color='gray', linestyle='--', alpha=0.5)
    
    ax2.set_xlabel('r (Å)', fontsize=12)
    ax2.set_ylabel('g(r)', fontsize=12)
    ax2.set_title('B) RDF - Distance centre de masse', fontsize=13)
    ax2.legend(fontsize=8, ncol=2, loc='upper right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 25)
    ax2.set_ylim(0, None)
    
    plt.suptitle('Fonctions de distribution radiale entre analogues', fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_rdf(all_results, FIGURES_DIR / 'rdf_comparison.png')

### 10.6 Comparaison des trois méthodes de détection

Comparaison directe des contacts détectés par les trois méthodes :
1. Distance minimale atomes lourds (notre implémentation)
2. Distance COM (notre implémentation)
3. Biopython NeighborSearch

In [ ]:
# =============================================================================
# Visualisation: Comparaison des trois méthodes de détection
# =============================================================================

def plot_method_comparison(all_results: Dict, cutoff: float = 4.0, 
                           save_path: Optional[Path] = None):
    """
    Compare les trois méthodes de détection de contacts.
    """
    # Préparer les données
    data = []
    for system_name, results in all_results.items():
        info = results['info']
        
        # Méthode 1: Distance min atomes lourds (avec cutoff spécifié)
        df_min = results['cutoff_sweep_min_heavy']
        n_contacts_min = df_min[df_min['cutoff'] == cutoff]['n_contacts'].mean()
        
        # Méthode 2: Distance COM (avec cutoff adaptatif)
        df_com = results['cutoff_sweep_com']
        adaptive_cutoff = compute_adaptive_cutoff(system_name)
        # Trouver le cutoff le plus proche
        available_cutoffs = df_com['cutoff'].unique()
        closest_cutoff = available_cutoffs[np.argmin(np.abs(available_cutoffs - adaptive_cutoff))]
        n_contacts_com = df_com[df_com['cutoff'] == closest_cutoff]['n_contacts'].mean()
        
        # Méthode 3: Biopython
        bp_res = results.get('biopython')
        n_contacts_bp = bp_res['mean_contacts'] if bp_res else 0
        
        data.append({
            'system': system_name.upper(),
            'aa_name': info[1],
            'aa_type': info[2],
            'min_heavy': n_contacts_min,
            'com': n_contacts_com,
            'biopython': n_contacts_bp,
            'com_cutoff': closest_cutoff,
        })
    
    df = pd.DataFrame(data)
    
    # Graphique
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel A: Barres groupées
    ax1 = axes[0]
    systems = df['system'].values
    x = np.arange(len(systems))
    width = 0.25
    
    bars1 = ax1.bar(x - width, df['min_heavy'], width, 
                    label=f'Min heavy atoms ({cutoff} Å)', color=COLORS['vdw'], alpha=0.8)
    bars2 = ax1.bar(x, df['com'], width, 
                    label='COM (adaptive cutoff)', color=COLORS['pipi'], alpha=0.8)
    bars3 = ax1.bar(x + width, df['biopython'], width, 
                    label=f'Biopython ({cutoff} Å)', color=COLORS['hbond'], alpha=0.8)
    
    ax1.set_xlabel('Système', fontsize=12)
    ax1.set_ylabel('Nombre moyen de contacts', fontsize=12)
    ax1.set_title('A) Nombre de contacts par méthode', fontsize=13)
    ax1.set_xticks(x)
    ax1.set_xticklabels(systems, rotation=45, ha='right')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Panel B: Corrélation entre méthodes
    ax2 = axes[1]
    
    # Min heavy vs Biopython
    ax2.scatter(df['min_heavy'], df['biopython'], 
                label='Min heavy vs Biopython', color=COLORS['vdw'], s=100, alpha=0.7)
    
    # Min heavy vs COM
    ax2.scatter(df['min_heavy'], df['com'], 
                label='Min heavy vs COM', color=COLORS['pipi'], s=100, alpha=0.7, marker='s')
    
    # Ligne de référence y=x
    max_val = max(df['min_heavy'].max(), df['biopython'].max(), df['com'].max())
    ax2.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='y=x')
    
    # Annotations
    for _, row in df.iterrows():
        ax2.annotate(row['system'], (row['min_heavy'], row['biopython']),
                    fontsize=8, alpha=0.7)
    
    ax2.set_xlabel('Contacts (min heavy atoms)', fontsize=12)
    ax2.set_ylabel('Contacts (autre méthode)', fontsize=12)
    ax2.set_title('B) Corrélation entre méthodes', fontsize=13)
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')
    
    plt.suptitle('Comparaison des méthodes de détection de contacts', fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()
    
    return df


# Générer le graphique
if all_results:
    comparison_df = plot_method_comparison(all_results, cutoff=4.0, 
                                           save_path=FIGURES_DIR / 'method_comparison.png')

### 10.7 Heatmap des contacts par système et cutoff

Visualisation matricielle du nombre de contacts pour faciliter l'identification des cutoffs optimaux.

In [ ]:
# =============================================================================
# Visualisation: Heatmaps des contacts
# =============================================================================

def plot_contact_heatmaps(all_results: Dict, save_path: Optional[Path] = None):
    """
    Génère des heatmaps du nombre de contacts par système et cutoff.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Préparer les données pour la méthode min heavy
    systems = list(all_results.keys())
    cutoffs_min = CUTOFF_RANGES['min_heavy']
    
    matrix_min = np.zeros((len(systems), len(cutoffs_min)))
    for i, system_name in enumerate(systems):
        df = all_results[system_name]['cutoff_sweep_min_heavy']
        for j, cutoff in enumerate(cutoffs_min):
            matrix_min[i, j] = df[df['cutoff'] == cutoff]['n_contacts'].mean()
    
    # Heatmap pour distance min
    ax1 = axes[0]
    im1 = ax1.imshow(matrix_min, aspect='auto', cmap='YlOrRd',
                     extent=[cutoffs_min[0], cutoffs_min[-1], len(systems)-0.5, -0.5])
    ax1.set_yticks(range(len(systems)))
    ax1.set_yticklabels([s.upper() for s in systems])
    ax1.set_xlabel('Cutoff (Å)', fontsize=12)
    ax1.set_ylabel('Système', fontsize=12)
    ax1.set_title('A) Distance minimale atomes lourds', fontsize=13)
    ax1.axvline(DEFAULT_CUTOFFS['vdw'], color='white', linestyle='--', lw=2)
    plt.colorbar(im1, ax=ax1, label='Nombre de contacts', shrink=0.8)
    
    # Préparer les données pour la méthode COM
    cutoffs_com = CUTOFF_RANGES['com']
    matrix_com = np.zeros((len(systems), len(cutoffs_com)))
    for i, system_name in enumerate(systems):
        df = all_results[system_name]['cutoff_sweep_com']
        for j, cutoff in enumerate(cutoffs_com):
            matrix_com[i, j] = df[df['cutoff'] == cutoff]['n_contacts'].mean()
    
    # Heatmap pour COM
    ax2 = axes[1]
    im2 = ax2.imshow(matrix_com, aspect='auto', cmap='YlOrRd',
                     extent=[cutoffs_com[0], cutoffs_com[-1], len(systems)-0.5, -0.5])
    ax2.set_yticks(range(len(systems)))
    ax2.set_yticklabels([s.upper() for s in systems])
    ax2.set_xlabel('Cutoff (Å)', fontsize=12)
    ax2.set_ylabel('Système', fontsize=12)
    ax2.set_title('B) Distance centre de masse', fontsize=13)
    plt.colorbar(im2, ax=ax2, label='Nombre de contacts', shrink=0.8)
    
    # Ajouter les cutoffs adaptatifs pour chaque système
    for i, system_name in enumerate(systems):
        adaptive = compute_adaptive_cutoff(system_name)
        ax2.plot(adaptive, i, 'w*', markersize=10)
    
    plt.suptitle('Heatmap: Nombre de contacts en fonction du système et du cutoff', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_contact_heatmaps(all_results, FIGURES_DIR / 'contact_heatmaps.png')

## 11. Analyse statistique et recommandations de cutoffs

### 11.1 Identification des cutoffs optimaux

Analyse quantitative pour identifier les cutoffs minimisant les faux positifs tout en conservant une sensibilité adéquate.

In [ ]:
# =============================================================================
# Analyse statistique et recommandations de cutoffs
# =============================================================================

def analyze_cutoff_statistics(all_results: Dict, target_fraction: float = 0.05) -> pd.DataFrame:
    """
    Analyse les statistiques des cutoffs pour identifier les valeurs optimales.
    
    Le cutoff optimal est défini comme la valeur minimale donnant une fraction
    de contacts inférieure à la cible (par défaut 5%).
    
    Parameters
    ----------
    all_results : Dict
        Résultats de l'analyse
    target_fraction : float
        Fraction cible de paires en contact (défaut: 5%)
        
    Returns
    -------
    pd.DataFrame
        Tableau des cutoffs recommandés par système
    """
    recommendations = []
    
    for system_name, results in all_results.items():
        info = results['info']
        
        # Méthode min heavy: trouver le cutoff donnant ~target_fraction
        df_min = results['cutoff_sweep_min_heavy'].copy()
        df_min['fraction'] = df_min['n_contacts'] / df_min['n_pairs']
        mean_frac_min = df_min.groupby('cutoff')['fraction'].mean()
        
        # Cutoff où fraction < target
        cutoff_min_optimal = mean_frac_min.index[mean_frac_min > target_fraction].min()
        if pd.isna(cutoff_min_optimal):
            cutoff_min_optimal = mean_frac_min.index[-1]
        
        # Fraction à 4.0 Å (cutoff VdW standard)
        frac_at_4A = mean_frac_min.get(4.0, 0)
        
        # Méthode COM
        df_com = results['cutoff_sweep_com'].copy()
        df_com['fraction'] = df_com['n_contacts'] / df_com['n_pairs']
        mean_frac_com = df_com.groupby('cutoff')['fraction'].mean()
        
        cutoff_com_optimal = mean_frac_com.index[mean_frac_com > target_fraction].min()
        if pd.isna(cutoff_com_optimal):
            cutoff_com_optimal = mean_frac_com.index[-1]
        
        # Cutoff adaptatif théorique
        adaptive_cutoff = compute_adaptive_cutoff(system_name)
        frac_at_adaptive = mean_frac_com.get(
            mean_frac_com.index[np.argmin(np.abs(mean_frac_com.index - adaptive_cutoff))], 0
        )
        
        recommendations.append({
            'Système': system_name.upper(),
            'Acide aminé': info[1],
            'Type': info[2],
            'Cutoff min_heavy (Å)': f"{cutoff_min_optimal:.2f}",
            'Frac @ 4.0Å (%)': f"{frac_at_4A*100:.1f}",
            'Cutoff COM (Å)': f"{cutoff_com_optimal:.1f}",
            'Cutoff adaptatif (Å)': f"{adaptive_cutoff:.1f}",
            'Frac @ adaptatif (%)': f"{frac_at_adaptive*100:.1f}",
        })
    
    return pd.DataFrame(recommendations)


# Générer le tableau de recommandations
if all_results:
    print("=" * 80)
    print("ANALYSE DES CUTOFFS OPTIMAUX")
    print("=" * 80)
    print(f"\nCritère: Cutoff minimal tel que fraction de paires en contact > 5%")
    print("(Indication du premier cutoff où les contacts deviennent significatifs)\n")
    
    recommendations_df = analyze_cutoff_statistics(all_results, target_fraction=0.05)
    display(recommendations_df.style.set_caption(
        "Recommandations de cutoffs par système"
    ))

### 11.2 Dérivée du nombre de contacts vs cutoff

La dérivée du nombre de contacts par rapport au cutoff permet d'identifier les régions de transition où le nombre de contacts augmente rapidement (potentiellement des faux positifs).

In [ ]:
# =============================================================================
# Analyse de la dérivée du nombre de contacts
# =============================================================================

def plot_contact_derivative(all_results: Dict, save_path: Optional[Path] = None):
    """
    Trace la dérivée du nombre de contacts par rapport au cutoff.
    Permet d'identifier les régions de transition.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_results)))
    
    # Panel A: Distance min heavy
    ax1 = axes[0]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_min_heavy']
        mean_contacts = df.groupby('cutoff')['n_contacts'].mean()
        
        # Calcul de la dérivée (différences finies)
        cutoffs = mean_contacts.index.values
        contacts = mean_contacts.values
        derivative = np.gradient(contacts, cutoffs)
        
        ax1.plot(cutoffs, derivative, label=system_name.upper(), 
                 color=colors[idx], lw=1.5)
    
    ax1.axvline(DEFAULT_CUTOFFS['vdw'], color='red', linestyle='--', lw=2)
    ax1.set_xlabel('Cutoff (Å)', fontsize=12)
    ax1.set_ylabel('dN/d(cutoff) (contacts/Å)', fontsize=12)
    ax1.set_title('A) Dérivée - Distance min atomes lourds', fontsize=13)
    ax1.legend(fontsize=8, ncol=2, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(2.5, 8)
    
    # Panel B: Distance COM
    ax2 = axes[1]
    for idx, (system_name, results) in enumerate(all_results.items()):
        df = results['cutoff_sweep_com']
        mean_contacts = df.groupby('cutoff')['n_contacts'].mean()
        
        cutoffs = mean_contacts.index.values
        contacts = mean_contacts.values
        derivative = np.gradient(contacts, cutoffs)
        
        ax2.plot(cutoffs, derivative, label=system_name.upper(),
                 color=colors[idx], lw=1.5)
    
    ax2.set_xlabel('Cutoff (Å)', fontsize=12)
    ax2.set_ylabel('dN/d(cutoff) (contacts/Å)', fontsize=12)
    ax2.set_title('B) Dérivée - Distance COM', fontsize=13)
    ax2.legend(fontsize=8, ncol=2, loc='upper right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(4, 15)
    
    plt.suptitle('Dérivée du nombre de contacts vs. cutoff\n'
                 '(Pics indiquent les régions de transition rapide)', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Figure sauvegardée: {save_path}")
    
    plt.show()


# Générer le graphique
if all_results:
    plot_contact_derivative(all_results, FIGURES_DIR / 'contact_derivative.png')

## 12. Export des résultats

### 12.1 Sauvegarde des données tabulaires

Export des résultats sous forme de fichiers CSV pour analyse ultérieure et archivage.

In [ ]:
# =============================================================================
# Export des résultats en CSV
# =============================================================================

def export_results_to_csv(all_results: Dict, output_dir: Path):
    """
    Exporte tous les résultats en fichiers CSV structurés.
    
    Parameters
    ----------
    all_results : Dict
        Dictionnaire complet des résultats
    output_dir : Path
        Répertoire de sortie
    """
    print("Export des résultats en CSV...")
    
    # 1. Balayage des cutoffs - Distance minimale
    print("  [1/7] Balayage cutoffs (min heavy)...", end=" ")
    all_cutoff_min = []
    for system_name, results in all_results.items():
        df = results['cutoff_sweep_min_heavy'].copy()
        df['system'] = system_name
        all_cutoff_min.append(df)
    
    df_all_min = pd.concat(all_cutoff_min, ignore_index=True)
    df_all_min.to_csv(output_dir / 'cutoff_sweep_min_heavy.csv', index=False)
    print("✓")
    
    # 2. Balayage des cutoffs - COM
    print("  [2/7] Balayage cutoffs (COM)...", end=" ")
    all_cutoff_com = []
    for system_name, results in all_results.items():
        df = results['cutoff_sweep_com'].copy()
        df['system'] = system_name
        all_cutoff_com.append(df)
    
    df_all_com = pd.concat(all_cutoff_com, ignore_index=True)
    df_all_com.to_csv(output_dir / 'cutoff_sweep_com.csv', index=False)
    print("✓")
    
    # 3. Comparaison des interactions
    print("  [3/7] Comparaison interactions...", end=" ")
    if 'interaction_df' in globals():
        interaction_df.to_csv(output_dir / 'interaction_comparison.csv', index=False)
    print("✓")
    
    # 4. Comparaison des méthodes
    print("  [4/7] Comparaison méthodes...", end=" ")
    if 'comparison_df' in globals():
        comparison_df.to_csv(output_dir / 'method_comparison.csv', index=False)
    print("✓")
    
    # 5. Recommandations de cutoffs
    print("  [5/7] Recommandations cutoffs...", end=" ")
    if 'recommendations_df' in globals():
        recommendations_df.to_csv(output_dir / 'cutoff_recommendations.csv', index=False)
    print("✓")
    
    # 6. RDF pour chaque système
    print("  [6/7] RDF...", end=" ")
    for system_name, results in all_results.items():
        # RDF COM
        rdf_com = results['rdf_com']
        df_rdf_com = pd.DataFrame({
            'r': rdf_com['r'],
            'g_r': rdf_com['g_r'],
            'g_r_std': rdf_com['g_r_std'],
            'g_r_sem': rdf_com['g_r_sem'],
        })
        df_rdf_com.to_csv(output_dir / f'rdf_com_{system_name}.csv', index=False)
        
        # RDF min heavy
        rdf_min = results['rdf_min_heavy']
        df_rdf_min = pd.DataFrame({
            'r': rdf_min['r'],
            'g_r': rdf_min['g_r'],
            'g_r_std': rdf_min['g_r_std'],
            'g_r_sem': rdf_min['g_r_sem'],
        })
        df_rdf_min.to_csv(output_dir / f'rdf_min_heavy_{system_name}.csv', index=False)
    print("✓")
    
    # 7. Résumé global
    print("  [7/7] Résumé global...", end=" ")
    summary_data = []
    for system_name, results in all_results.items():
        info = results['info']
        
        # Statistiques de base
        n_frames = results['n_frames']
        n_analogues = results['n_analogues']
        
        # Contacts moyens par type (avec cutoffs par défaut)
        interactions = results['interactions']
        n_vdw = np.mean([r['n_contacts'] for r in interactions['vdw']])
        n_hbond = np.mean([r['n_contacts'] for r in interactions['hbond']])
        n_hydro = np.mean([r['n_contacts'] for r in interactions['hydrophobic']])
        n_pipi = np.mean([r['n_contacts'] for r in interactions['pipi']])
        
        # Biopython
        bp = results.get('biopython')
        n_bp = bp['mean_contacts'] if bp else 0
        
        summary_data.append({
            'system': system_name,
            'residue_name': info[0],
            'amino_acid': info[1],
            'type': info[2],
            'n_frames': n_frames,
            'n_analogues_per_frame': n_analogues,
            'contacts_vdw_mean': n_vdw,
            'contacts_hbond_mean': n_hbond,
            'contacts_hydrophobic_mean': n_hydro,
            'contacts_pipi_mean': n_pipi,
            'contacts_biopython_mean': n_bp,
            'cutoff_vdw': DEFAULT_CUTOFFS['vdw'],
            'cutoff_hbond': DEFAULT_CUTOFFS['hbond'],
            'cutoff_hydrophobic': DEFAULT_CUTOFFS['hydrophobic'],
            'cutoff_pipi': DEFAULT_CUTOFFS['pipi'],
        })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv(output_dir / 'summary_global.csv', index=False)
    print("✓")
    
    print(f"\n✓ Export terminé: {output_dir}")
    return df_summary


# Export des résultats
if all_results:
    summary_global = export_results_to_csv(all_results, OUTPUT_DIR)

### 12.2 Export au format JSON

Sauvegarde des métadonnées et résultats structurés au format JSON pour interopérabilité.

In [ ]:
# =============================================================================
# Export des métadonnées au format JSON
# =============================================================================

import json
from datetime import datetime

def export_metadata_to_json(all_results: Dict, output_dir: Path):
    """
    Exporte les métadonnées et résultats agrégés au format JSON.
    
    Parameters
    ----------
    all_results : Dict
        Dictionnaire complet des résultats
    output_dir : Path
        Répertoire de sortie
    """
    print("Export des métadonnées JSON...")
    
    metadata = {
        'analysis_info': {
            'date': datetime.now().isoformat(),
            'notebook': 'analogue_interactions_analysis.ipynb',
            'version': '1.0',
            'description': 'Analyse quantitative des interactions entre analogues de chaînes latérales',
        },
        'parameters': {
            'default_cutoffs': DEFAULT_CUTOFFS,
            'cutoff_ranges': {
                'min_heavy': [float(CUTOFF_RANGES['min_heavy'][0]), 
                             float(CUTOFF_RANGES['min_heavy'][-1])],
                'com': [float(CUTOFF_RANGES['com'][0]), 
                       float(CUTOFF_RANGES['com'][-1])],
            },
            'rdf_parameters': {
                'r_max': 30.0,
                'dr': 0.1,
            },
        },
        'systems': {},
    }
    
    # Ajouter les informations par système
    for system_name, results in all_results.items():
        info = results['info']
        interactions = results['interactions']
        bp = results.get('biopython')
        
        system_meta = {
            'residue_name': info[0],
            'amino_acid': info[1],
            'type': info[2],
            'n_frames': results['n_frames'],
            'n_analogues_per_frame': results['n_analogues'],
            'contacts': {
                'vdw': {
                    'cutoff': DEFAULT_CUTOFFS['vdw'],
                    'mean': float(np.mean([r['n_contacts'] for r in interactions['vdw']])),
                    'std': float(np.std([r['n_contacts'] for r in interactions['vdw']])),
                },
                'hbond': {
                    'cutoff': DEFAULT_CUTOFFS['hbond'],
                    'mean': float(np.mean([r['n_contacts'] for r in interactions['hbond']])),
                    'std': float(np.std([r['n_contacts'] for r in interactions['hbond']])),
                },
                'hydrophobic': {
                    'cutoff': DEFAULT_CUTOFFS['hydrophobic'],
                    'mean': float(np.mean([r['n_contacts'] for r in interactions['hydrophobic']])),
                    'std': float(np.std([r['n_contacts'] for r in interactions['hydrophobic']])),
                },
                'pipi': {
                    'cutoff': DEFAULT_CUTOFFS['pipi'],
                    'mean': float(np.mean([r['n_contacts'] for r in interactions['pipi']])),
                    'std': float(np.std([r['n_contacts'] for r in interactions['pipi']])),
                },
                'biopython': {
                    'cutoff': 4.0,
                    'mean': float(bp['mean_contacts']) if bp else 0.0,
                    'std': float(bp['std_contacts']) if bp else 0.0,
                },
            },
            'rdf': {
                'first_peak_com': {
                    'position': float(results['rdf_com']['r'][np.argmax(results['rdf_com']['g_r'][:50])]),
                    'height': float(results['rdf_com']['g_r'][:50].max()),
                },
                'first_peak_min_heavy': {
                    'position': float(results['rdf_min_heavy']['r'][np.argmax(results['rdf_min_heavy']['g_r'][:50])]),
                    'height': float(results['rdf_min_heavy']['g_r'][:50].max()),
                },
            },
            'adaptive_cutoff': float(compute_adaptive_cutoff(system_name)),
        }
        
        metadata['systems'][system_name] = system_meta
    
    # Sauvegarder
    json_path = output_dir / 'analysis_metadata.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Métadonnées exportées: {json_path}")
    
    return metadata


# Export JSON
if all_results:
    metadata = export_metadata_to_json(all_results, OUTPUT_DIR)

## 13. Résumé et conclusions techniques

### 13.1 Tableau récapitulatif des résultats

Synthèse des résultats clés pour tous les systèmes analysés.

In [ ]:
# =============================================================================
# Tableau récapitulatif final
# =============================================================================

def generate_final_summary(all_results: Dict) -> pd.DataFrame:
    """
    Génère un tableau récapitulatif complet des résultats.
    
    Parameters
    ----------
    all_results : Dict
        Dictionnaire complet des résultats
        
    Returns
    -------
    pd.DataFrame
        Tableau récapitulatif
    """
    summary_rows = []
    
    for system_name, results in all_results.items():
        info = results['info']
        interactions = results['interactions']
        
        # Contacts moyens par type d'interaction
        n_vdw = np.mean([r['n_contacts'] for r in interactions['vdw']])
        n_hbond = np.mean([r['n_contacts'] for r in interactions['hbond']])
        n_hydro = np.mean([r['n_contacts'] for r in interactions['hydrophobic']])
        n_pipi = np.mean([r['n_contacts'] for r in interactions['pipi']])
        
        # Fractions
        frac_vdw = np.mean([r['fraction_in_contact'] for r in interactions['vdw']])
        frac_hbond = np.mean([r['fraction_in_contact'] for r in interactions['hbond']])
        frac_hydro = np.mean([r['fraction_in_contact'] for r in interactions['hydrophobic']])
        frac_pipi = np.mean([r['fraction_in_contact'] for r in interactions['pipi']])
        
        # RDF - position du premier pic
        rdf_com = results['rdf_com']
        rdf_min = results['rdf_min_heavy']
        
        # Trouver le premier pic (r < 10 Å)
        mask_com = rdf_com['r'] < 10
        mask_min = rdf_min['r'] < 10
        
        first_peak_com = rdf_com['r'][mask_com][np.argmax(rdf_com['g_r'][mask_com])]
        first_peak_min = rdf_min['r'][mask_min][np.argmax(rdf_min['g_r'][mask_min])]
        
        # Nombre de coordination (à 6 Å)
        coord_num_com = compute_coordination_number(rdf_com, r_cutoff=6.0)
        coord_num_min = compute_coordination_number(rdf_min, r_cutoff=6.0)
        
        summary_rows.append({
            'Système': system_name.upper(),
            'Acide aminé': info[1],
            'Type': info[2],
            'N frames': results['n_frames'],
            'N analogues': results['n_analogues'],
            
            # Contacts moyens
            'VdW': f"{n_vdw:.1f}",
            'H-bond': f"{n_hbond:.1f}",
            'Hydrophobe': f"{n_hydro:.1f}",
            'π-π': f"{n_pipi:.1f}",
            
            # Fractions (%)
            'VdW (%)': f"{frac_vdw*100:.1f}",
            'H-bond (%)': f"{frac_hbond*100:.1f}",
            'Hydro (%)': f"{frac_hydro*100:.1f}",
            'π-π (%)': f"{frac_pipi*100:.1f}",
            
            # RDF
            '1er pic COM (Å)': f"{first_peak_com:.2f}",
            '1er pic min (Å)': f"{first_peak_min:.2f}",
            'N coord COM': f"{coord_num_com:.2f}",
            'N coord min': f"{coord_num_min:.2f}",
        })
    
    return pd.DataFrame(summary_rows)


# Générer et afficher le résumé final
if all_results:
    print("=" * 100)
    print("TABLEAU RÉCAPITULATIF FINAL")
    print("=" * 100)
    print()
    
    final_summary = generate_final_summary(all_results)
    
    # Affichage stylisé
    display(final_summary.style\
        .set_caption("Résumé complet de l'analyse des interactions entre analogues")\
        .background_gradient(subset=['VdW', 'H-bond', 'Hydrophobe', 'π-π'], cmap='YlOrRd')\
        .format(precision=1))
    
    # Sauvegarder
    final_summary.to_csv(OUTPUT_DIR / 'final_summary.csv', index=False)
    print(f"\n✓ Résumé sauvegardé: {OUTPUT_DIR / 'final_summary.csv'}")

### 13.2 Statistiques globales et observations techniques

In [ ]:
# =============================================================================
# Statistiques globales
# =============================================================================

print("=" * 100)
print("STATISTIQUES GLOBALES DE L'ANALYSE")
print("=" * 100)

# Compter les systèmes par type d'acide aminé
aa_types = {}
for system_name, results in all_results.items():
    aa_type = results['info'][2]
    aa_types[aa_type] = aa_types.get(aa_type, 0) + 1

print("\n1. DISTRIBUTION DES SYSTÈMES PAR TYPE")
print("─" * 50)
for aa_type, count in sorted(aa_types.items()):
    print(f"  {aa_type.capitalize():15s}: {count:2d} système(s)")

# Statistiques sur les contacts
print("\n2. STATISTIQUES DES CONTACTS (avec cutoffs par défaut)")
print("─" * 50)

all_vdw = []
all_hbond = []
all_hydro = []
all_pipi = []

for results in all_results.values():
    interactions = results['interactions']
    all_vdw.extend([r['n_contacts'] for r in interactions['vdw']])
    all_hbond.extend([r['n_contacts'] for r in interactions['hbond']])
    all_hydro.extend([r['n_contacts'] for r in interactions['hydrophobic']])
    all_pipi.extend([r['n_contacts'] for r in interactions['pipi']])

print(f"\n  Van der Waals (cutoff = {DEFAULT_CUTOFFS['vdw']} Å):")
print(f"    Moyenne: {np.mean(all_vdw):.2f} ± {np.std(all_vdw):.2f} contacts/frame")
print(f"    Médiane: {np.median(all_vdw):.2f}, Min: {np.min(all_vdw):.0f}, Max: {np.max(all_vdw):.0f}")

print(f"\n  Liaisons H (cutoff = {DEFAULT_CUTOFFS['hbond']} Å):")
print(f"    Moyenne: {np.mean(all_hbond):.2f} ± {np.std(all_hbond):.2f} contacts/frame")
print(f"    Médiane: {np.median(all_hbond):.2f}, Min: {np.min(all_hbond):.0f}, Max: {np.max(all_hbond):.0f}")

print(f"\n  Hydrophobes (cutoff = {DEFAULT_CUTOFFS['hydrophobic']} Å):")
print(f"    Moyenne: {np.mean(all_hydro):.2f} ± {np.std(all_hydro):.2f} contacts/frame")
print(f"    Médiane: {np.median(all_hydro):.2f}, Min: {np.min(all_hydro):.0f}, Max: {np.max(all_hydro):.0f}")

print(f"\n  Interactions π-π (cutoff = {DEFAULT_CUTOFFS['pipi']} Å):")
print(f"    Moyenne: {np.mean(all_pipi):.2f} ± {np.std(all_pipi):.2f} contacts/frame")
print(f"    Médiane: {np.median(all_pipi):.2f}, Min: {np.min(all_pipi):.0f}, Max: {np.max(all_pipi):.0f}")

# Comparaison des méthodes
print("\n3. COMPARAISON DES MÉTHODES DE DÉTECTION (cutoff = 4.0 Å)")
print("─" * 50)

methods_comparison = []
for system_name, results in all_results.items():
    df_min = results['cutoff_sweep_min_heavy']
    n_min = df_min[df_min['cutoff'] == 4.0]['n_contacts'].mean()
    
    bp = results.get('biopython')
    n_bp = bp['mean_contacts'] if bp else 0
    
    methods_comparison.append({
        'min_heavy': n_min,
        'biopython': n_bp,
        'diff': abs(n_min - n_bp),
        'rel_diff': abs(n_min - n_bp) / max(n_min, 0.1) * 100,
    })

print(f"\n  Méthode 1 (min heavy atoms):")
print(f"    Moyenne: {np.mean([m['min_heavy'] for m in methods_comparison]):.2f} contacts/frame")

print(f"\n  Méthode 3 (Biopython NeighborSearch):")
print(f"    Moyenne: {np.mean([m['biopython'] for m in methods_comparison]):.2f} contacts/frame")

print(f"\n  Différence absolue moyenne: {np.mean([m['diff'] for m in methods_comparison]):.2f} contacts")
print(f"  Différence relative moyenne: {np.mean([m['rel_diff'] for m in methods_comparison]):.1f}%")

# Corrélation Pearson et Spearman
min_heavy_vals = [m['min_heavy'] for m in methods_comparison]
bp_vals = [m['biopython'] for m in methods_comparison]

if len(min_heavy_vals) > 2:
    r_pearson, p_pearson = pearsonr(min_heavy_vals, bp_vals)
    r_spearman, p_spearman = spearmanr(min_heavy_vals, bp_vals)
    
    print(f"\n  Corrélation Pearson: r = {r_pearson:.3f} (p = {p_pearson:.3e})")
    print(f"  Corrélation Spearman: ρ = {r_spearman:.3f} (p = {p_spearman:.3e})")

# Données totales traitées
print("\n4. VOLUME DE DONNÉES TRAITÉES")
print("─" * 50)

total_frames = sum(r['n_frames'] for r in all_results.values())
total_pdb_files = len([f for files in pdb_systems.values() for f in files])
total_analogues = sum(r['n_frames'] * r['n_analogues'] for r in all_results.values())

print(f"\n  Systèmes analysés: {len(all_results)}")
print(f"  Fichiers PDB traités: {total_pdb_files}")
print(f"  Frames analysées: {total_frames}")
print(f"  Analogues analysés (total): {total_analogues}")
print(f"  Paires d'analogues évaluées: ~{total_analogues * (total_analogues - 1) // 2:,}")

print("\n" + "=" * 100)

### 13.3 Fichiers générés et utilisation

Liste des fichiers de sortie produits par cette analyse.

In [ ]:
# =============================================================================
# Inventaire des fichiers générés
# =============================================================================

print("=" * 100)
print("FICHIERS GÉNÉRÉS")
print("=" * 100)

print("\n📊 FIGURES (PNG, 300 DPI)")
print("─" * 50)

figures_list = [
    ('distance_distributions.png', 'Distributions des distances inter-analogues'),
    ('cutoff_sweep.png', 'Balayage des cutoffs (2 méthodes)'),
    ('contact_fraction.png', 'Fraction de paires en contact vs cutoff'),
    ('interaction_comparison.png', 'Comparaison des types d\'interactions'),
    ('rdf_comparison.png', 'Fonctions de distribution radiale (RDF)'),
    ('method_comparison.png', 'Comparaison des 3 méthodes de détection'),
    ('contact_heatmaps.png', 'Heatmaps nombre de contacts vs cutoff'),
    ('contact_derivative.png', 'Dérivée du nombre de contacts'),
]

for filename, description in figures_list:
    fig_path = FIGURES_DIR / filename
    if fig_path.exists():
        size_mb = fig_path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {filename:35s} - {description} ({size_mb:.2f} MB)")
    else:
        print(f"  ○ {filename:35s} - {description} (non généré)")

print("\n📁 DONNÉES CSV")
print("─" * 50)

csv_files = [
    ('cutoff_sweep_min_heavy.csv', 'Balayage cutoffs - distance min atomes lourds'),
    ('cutoff_sweep_com.csv', 'Balayage cutoffs - distance COM'),
    ('interaction_comparison.csv', 'Comparaison des types d\'interactions'),
    ('method_comparison.csv', 'Comparaison des méthodes de détection'),
    ('cutoff_recommendations.csv', 'Recommandations de cutoffs optimaux'),
    ('summary_global.csv', 'Résumé global par système'),
    ('final_summary.csv', 'Tableau récapitulatif final complet'),
]

for filename, description in csv_files:
    csv_path = OUTPUT_DIR / filename
    if csv_path.exists():
        n_rows = sum(1 for _ in open(csv_path)) - 1  # Sans le header
        print(f"  ✓ {filename:35s} - {description} ({n_rows} lignes)")
    else:
        print(f"  ○ {filename:35s} - {description} (non généré)")

# RDF individuelles
print("\n  RDF par système:")
for system_name in all_results.keys():
    rdf_com_path = OUTPUT_DIR / f'rdf_com_{system_name}.csv'
    rdf_min_path = OUTPUT_DIR / f'rdf_min_heavy_{system_name}.csv'
    if rdf_com_path.exists() and rdf_min_path.exists():
        print(f"    ✓ {system_name.upper():10s} - rdf_com_{system_name}.csv, rdf_min_heavy_{system_name}.csv")

print("\n📋 MÉTADONNÉES JSON")
print("─" * 50)

json_path = OUTPUT_DIR / 'analysis_metadata.json'
if json_path.exists():
    size_kb = json_path.stat().st_size / 1024
    print(f"  ✓ analysis_metadata.json - Métadonnées complètes de l'analyse ({size_kb:.1f} KB)")
else:
    print(f"  ○ analysis_metadata.json - (non généré)")

# Emplacement des fichiers
print("\n📂 EMPLACEMENTS")
print("─" * 50)
print(f"\n  Figures:  {FIGURES_DIR}")
print(f"  Données:  {OUTPUT_DIR}")

# Instructions d'utilisation
print("\n" + "=" * 100)
print("UTILISATION DES RÉSULTATS")
print("=" * 100)
print("""
Les fichiers CSV peuvent être importés dans:
  • Excel/LibreOffice pour analyse tabulaire
  • Python/pandas pour analyses complémentaires
  • R pour analyses statistiques avancées
  • GraphPad Prism pour publication

Les figures PNG sont prêtes pour:
  • Inclusion dans présentations (PowerPoint, Keynote)
  • Intégration dans manuscrits
  • Analyse visuelle comparative

Le fichier JSON contient:
  • Métadonnées complètes de l'analyse
  • Paramètres utilisés
  • Résultats agrégés par système
  • Compatible avec outils d'analyse automatique

Les RDF peuvent être comparées avec:
  • Calculs NAMD (format compatible, dr = 0.1 Å)
  • Données expérimentales (diffusion)
  • Autres simulations MD
""")

---

## Notes méthodologiques finales

### Points clés de l'implémentation

1. **Deux définitions du contact** implémentées et comparées systématiquement :
   - Distance minimale entre atomes lourds (approche atomique fine)
   - Distance entre centres de masse (approche résidu-résidu)

2. **Quatre types d'interactions** caractérisés quantitativement :
   - Van der Waals : distance inter-atomique ≤ 4.0 Å
   - Liaisons hydrogène : distance donneur-accepteur ≤ 3.5 Å
   - Interactions hydrophobes : distance C-C ≤ 4.5 Å
   - Interactions π-π : distance centroïde-centroïde ≤ 5.0 Å

3. **Validation croisée** avec Biopython NeighborSearch pour vérifier la cohérence des résultats

4. **Analyse RDF** compatible NAMD (dr = 0.1 Å) pour frames indépendantes (non trajectoire continue)

5. **Balayage paramétrique** exhaustif des cutoffs pour identification objective des seuils optimaux

### Limitations techniques

- Les frames PDB sont **indépendantes** (non corrélées temporellement) : pas de dynamique temporelle
- Les RDF sont agrégées sur les frames disponibles sans pondération temporelle
- Les critères géométriques (angles D-H-A pour H-bonds) sont simplifiés
- Pas de prise en compte des conditions périodiques de boîte (PBC) dans les calculs de distance

### Recommandations pour l'analyse

- Comparer les **trois méthodes** (min heavy, COM, Biopython) pour valider la robustesse
- Utiliser les **courbes de dérivée** pour identifier les transitions de régime
- Considérer la **fraction de contacts** plutôt que le nombre absolu pour comparaisons inter-systèmes
- Examiner les **RDF** pour détecter une structuration spatiale éventuelle

### Adaptabilité du code

Le notebook est conçu pour être facilement adapté :
- Ajout de nouveaux systèmes : détection automatique des dossiers `sc*`
- Modification des cutoffs : paramètres centralisés dans `DEFAULT_CUTOFFS`
- Extension à d'autres interactions : structure modulaire des fonctions de détection
- Application à d'autres membranes : modification des sélections dans `load_frame_mda`

---

**Fin de l'analyse**

*Notebook généré le : décembre 2025*  
*Version : 1.0*  
*Auteur : Analyse automatisée des interactions entre analogues*

## 12. Export des résultats

### 12.1 Sauvegarde des données brutes et résumés